<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/flashvsr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture
# title: Install
# One cell: the FlashVSR fork, its weights, the media helpers and the VRAM planner.
#
#   1. pip install the things Colab's image is missing (einops, imageio-ffmpeg, a video decoder). No
#      Block-Sparse-Attention, no flash-attn, no diffsynth package: this fork vendors its own src/ and
#      can run attention through plain torch SDPA, which is what a T4 has.
#   2. git clone --depth 1 the self-contained ComfyUI-FlashVSR_Stable fork into /content/FlashVSR
#      (it ships nodes.py, cli_main.py, src/, model_paths.yaml and posi_prompt.pth), then import it once
#      to prove the copy works and to read the exact kwargs its flashvsr() node accepts.
#   3. Download the weights into /content/models. The repos are PUBLIC (apache-2.0), so there is no
#      token to paste and no login cell; the cell verifies size per file and resumes a broken download.
#   4. Write the media helpers (probe, kind, size, duration) and the memory planner that Config shows.
#
# TWO THINGS THAT BITE PEOPLE ON COLAB
# There is no fp8/GGUF/4-bit FlashVSR — nothing to pick in a quantization table. What decides whether a
# run fits in 16 GB is the decoder (the default "full + Wan2.1 VAE" costs ~7.5 GB of weights), the two
# tiling switches, and the chunk size, which the Run cell fits by measuring. Config shows the arithmetic
# before you start, and the Run cell re-fits instead of dying when the card disagrees.
# /content dies with the runtime: idle ~90 minutes (12 h free / 24 h Pro at the wall), and Colab may
# recycle the instance at any moment. Weights are therefore re-downloaded in a new session (~7 GB, a few
# minutes), and finished files must leave through the Download cell before you close the tab — anything
# still in /content when the runtime dies is gone, and the resume file with it.
import os, re, sys, json, time, shutil, hashlib, subprocess

!nvidia-smi | head -14 || true        # which card this session actually got, in the runtime's own words

HF_TOKEN = ''        # every repo used here is public; nothing logs in, nothing is gated

class _V:
    """A value wearing the interface the rest of the notebook reads from a widget: `name.value`.

    This build has no widget boxes on purpose — Colab renders a `#@param` form better than any
    ipywidgets layout — but the Run and Download cells were written against widgets and read about forty
    settings. So the form fields are wrapped in these, and everything that is NOT offered keeps the name it
    had, with its default. It also means a scratch cell can override any of them (`chunk_w.value = 48`) and
    the next run obeys. `.options` is there because a couple of places ask what a field could have been.
    """
    __slots__ = ('value', 'options')

    def __init__(self, value, options=()):
        self.value = value
        self.options = tuple(options)

    def __repr__(self):
        return f'{self.value!r}'


WORK = '/content'
if not os.path.isdir(WORK):                       # lets the cells be exercised outside Colab
    WORK = os.path.join(os.path.expanduser('~'), 'colab_sim', 'content')
REPO_DIR = f'{WORK}/FlashVSR'
DRIVER = f'{WORK}/fvsr_batch.py'
INPUT_DIR = f'{WORK}/inputs'
OUTPUT_DIR = f'{WORK}/outputs'
SELECTED_JSON = f'{WORK}/selected_media.json'
STATE_PATH = f'{WORK}/fvsr_state.json'
SCRATCH = f'{WORK}/.fvsr'                        # queue links and per-process driver output
QUEUE_IN = f'{SCRATCH}/queue_in'
DRV_OUT = f'{SCRATCH}/drv_out'
for d in (INPUT_DIR, OUTPUT_DIR, QUEUE_IN, DRV_OUT):
    os.makedirs(d, exist_ok=True)
# 6.4-9.3 GB of weights, and /content is the only writable disk a Colab runtime gives you: they are as
# temporary as the session, so a fresh runtime pays for the download again unless the files are already
# there. That is also why the Run cell links its queue instead of copying media around.
MODELS_DIR = f'{WORK}/models'
os.makedirs(MODELS_DIR, exist_ok=True)
print(f'work {WORK}  ·  repo {REPO_DIR}  ·  weights {MODELS_DIR}  (ephemeral: re-downloaded in a new '
      f'session)  ·  results {OUTPUT_DIR}')

# ------------------------------------------------------------------ 1. packages
# torch/torchvision/PIL/numpy/tqdm/psutil/yaml/huggingface_hub are already on Colab's image.
# opencv is only needed by the fork's own CLI (this notebook drives nodes.py directly), and
# sageattention / flash-attn are optional accelerators - sdpa is asked for explicitly.
try:
    import einops        # noqa: F401
    _need_einops = []
except Exception:
    _need_einops = ['einops']
try:
    import imageio_ffmpeg    # noqa: F401
    _need_ff = []
except Exception:
    _need_ff = ['imageio-ffmpeg']
_pkgs = _need_einops + _need_ff
if _pkgs:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', *_pkgs], check=False)
    print('installed:', ' '.join(_pkgs) or '(nothing)')
else:
    print('python packages already present (einops, imageio-ffmpeg)')
try:
    import cv2       # noqa: F401
    print('opencv present (the fork CLI would also work)')
except Exception:
    print('note: no opencv — irrelevant here, this notebook does not use the fork\u2019s cv2 writer')

FFMPEG = shutil.which('ffmpeg')
FFPROBE = shutil.which('ffprobe')
if not FFMPEG:
    try:
        import imageio_ffmpeg
        FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
        os.environ['PATH'] = os.path.dirname(FFMPEG) + os.pathsep + os.environ.get('PATH', '')
        print('using the ffmpeg bundled with imageio-ffmpeg:', FFMPEG)
    except Exception:
        FFMPEG = None
print('ffmpeg:', FFMPEG or 'MISSING - install ffmpeg (the driver needs it to read video)')

# ------------------------------------------------------------------ 2. the fork
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('repo already cloned, updating…')
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--', 'model_paths.yaml'],
                   capture_output=True, text=True)      # this cell rewrites that file; let the pull work
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
    print((r.stdout or r.stderr).strip()[:200] or 'up to date')
else:
    print('git clone --depth 1 https://github.com/naxci1/ComfyUI-FlashVSR_Stable …')
    t0 = time.time()
    for attempt in (1, 2):
        r = subprocess.run(['git', 'clone', '--depth', '1', '--single-branch',
                            'https://github.com/naxci1/ComfyUI-FlashVSR_Stable', REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.isfile(f'{REPO_DIR}/nodes.py'):
            break
        print(f'clone attempt {attempt} failed: {(r.stderr or r.stdout).strip()[-300:]}')
        if attempt == 1:
            time.sleep(5)
print(f'cloned in {time.time()-t0:.0f}s')

for rel in ('nodes.py', 'posi_prompt.pth', 'model_paths.yaml', 'src/pipelines/flashvsr_tiny.py'):
    if not os.path.isfile(os.path.join(REPO_DIR, rel)):
        raise RuntimeError(f'the clone is missing {rel} — internet off, or the repo layout changed. '
                           'Check the two lines above.')
# Tell the fork where the weights are, using its own supported mechanism: model_paths.yaml wins over
# anything else it might guess (the driver also mocks folder_paths, but a stale ComfyUI config or a
# relative "models" default is exactly the kind of thing that makes a downloaded model "disappear").
_mp = os.path.join(REPO_DIR, 'model_paths.yaml')
with open(_mp, 'w', encoding='utf-8') as f:
    f.write('# written by flashvsr_image_video_upscale.ipynb — keep the fork from re-downloading weights\n'
            f'flashvsr_model_path: "{MODELS_DIR}"\n')
print(f'pointed {os.path.basename(_mp)} at {MODELS_DIR}')
head = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
size = sum(os.path.getsize(os.path.join(r, f)) for r, _d, fs in os.walk(REPO_DIR) for f in fs
           if os.path.exists(os.path.join(r, f)))
print(f'repo at {head}, {size/1e6:.1f} MB (weights not included)')

# ------------------------------------------------------------------ 3. weights
FV_DIT = 'diffusion_pytorch_model_streaming_dmd.safetensors'
FV_PARTS = {'dit': (FV_DIT, 5.676), 'lq': ('LQ_proj_in.ckpt', 0.576), 'tcd': ('TCDecoder.ckpt', 0.189),
            'vae_wan21': ('Wan2.1_VAE.pth', 0.508)}
FV_VAES = {'Wan2.1': ('JunhaoZhuang/FlashVSR-v1.1', 'Wan2.1_VAE.pth', 0.508),
           'Wan2.2': ('lightx2v/Autoencoders', 'Wan2.2_VAE.pth', 2.819),
           'LightVAE_W2.1': ('lightx2v/Autoencoders', 'lightvaew2_1.pth', 0.032),
           'TAE_W2.2': ('lightx2v/Autoencoders', 'taew2_2.pth', 0.023),
           'LightTAE_HY1.5': ('lightx2v/Autoencoders', 'lighttaehy1_5.pth', 0.046)}
FV_WEIGHTS = {m: {'repo': f'JunhaoZhuang/{m}', 'dir': os.path.join(MODELS_DIR, m), 'parts': FV_PARTS}
              for m in ('FlashVSR', 'FlashVSR-v1.1')}


def model_path(model, part):
    """Absolute path of one weight file. Everything - DiT, LQ_proj, TCDecoder and any VAE - lives inside
    <MODELS_DIR>/<model>/, because that is where nodes.py looks for it."""
    fname = FV_VAES[part][1] if part in FV_VAES else FV_PARTS[part][0]
    return os.path.join(FV_WEIGHTS[model]['dir'], fname)


def part_gb(part):
    return FV_VAES[part][2] if part in FV_VAES else FV_PARTS[part][1]


def model_present(model, part):
    p = model_path(model, part)
    return os.path.isfile(p) and os.path.getsize(p) > part_gb(part) * 1e9 * 0.98


def fmt_size(n_bytes):
    for u in ('B', 'KB', 'MB', 'GB', 'TB'):
        if abs(n_bytes) < 1024 or u == 'TB':
            return f'{n_bytes:.1f} {u}' if u != 'B' else f'{int(n_bytes)} B'
        n_bytes /= 1024.0
    return ''


def free_gb(path=MODELS_DIR):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return 999.0


def _hf_kwargs(fn, **want):
    """Pass only what this huggingface_hub actually accepts.

    Kaggle images ship different hub generations: on hub >= 1.0 `quiet` and `local_dir_use_symlinks`
    were REMOVED (a TypeError in the middle of a 5.7 GB download is a nasty way to learn it), on older
    ones they are the documented way. So the call is filtered through the live signature."""
    try:
        import inspect
        ok = set(inspect.signature(fn).parameters)
    except (TypeError, ValueError):
        return want
    drop = sorted(k for k in want if k not in ok)
    if drop:
        print('  note: this huggingface_hub has no ' + ', '.join(drop) + ' — not passed')
    return {k: v for k, v in want.items() if k in ok}


def _hf_download(repo_id, filename, dest_dir, quiet=False):
    """One file straight into dest_dir (local_dir, not cache_dir: no second copy on disk), resumable."""
    from huggingface_hub import hf_hub_download
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, filename)
    if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER', '').lower() in ('', '0', 'auto'):
        try:
            import hf_transfer                      # noqa: F401
            os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
        except Exception:
            os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    hint = os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1' and 'hf_transfer' or 'plain https'
    if not quiet:
        print(f'  downloading {filename} from {repo_id} ({hint})…')
    kw = _hf_kwargs(hf_hub_download, repo_id=repo_id, filename=filename, local_dir=dest_dir,
                    token=(HF_TOKEN or None))
    try:
        p = hf_hub_download(**kw)
    except TypeError as exc:
        # present in the signature but refused at runtime (deprecated-arg shims): fall back to a cache
        # download and move the file ourselves.
        print(f'  hub refused an argument ({exc}); retrying through the cache and copying')
        p = hf_hub_download(repo_id=repo_id, filename=filename, cache_dir=dest_dir)
    if not os.path.isfile(p):
        p = dest
    if os.path.isfile(p) and os.path.abspath(p) != os.path.abspath(dest):
        shutil.move(p, dest)                        # the fork looks for <models_dir>/<model>/<file>
    return dest


def fetch_weights(model, mode='tiny', vae='Wan2.1', quiet=False):
    """DiT + LQ_proj + TCDecoder, plus the chosen VAE when (and only when) mode is 'full'.
    tiny / tiny-long decode with TCDecoder, so downloading a VAE for them is 0.5-2.8 GB of nothing."""
    need = ['dit', 'lq', 'tcd'] + ([vae] if mode == 'full' else [])
    todo = [k for k in need if not model_present(model, k)]
    want_gb = sum(part_gb(k) for k in todo)
    if not quiet:
        print(f'{model} ({mode}): {len(todo)}/{len(need)} file(s) to fetch, ~{want_gb:.2f} GB')
    if want_gb and free_gb() < want_gb + 1.5:
        raise RuntimeError(f'only {free_gb():.1f} GB free in {MODELS_DIR} — {want_gb:.2f} GB of weights '
                           'will not fit. /content is the only writable disk here: clear old results '
                           'with the Upload cell\'s "Clear outputs" and run this cell again.')
    got = {}
    for k in todo:
        repo = FV_VAES[k][0] if k in FV_VAES else FV_WEIGHTS[model]['repo']
        fname = FV_VAES[k][1] if k in FV_VAES else FV_PARTS[k][0]
        dest = FV_WEIGHTS[model]['dir']       # the fork expects every file inside the model folder
        path = _hf_download(repo, fname, dest, quiet=quiet)
        expect = part_gb(k)
        real = os.path.getsize(path) if os.path.isfile(path) else 0
        if not os.path.isfile(path):
            raise RuntimeError(f'{fname} did not land in {dest} — huggingface_hub returned nothing '
                               'usable. Re-run this cell; if it repeats, copy the file in manually from '
                               'https://huggingface.co/' + repo)
        if real < expect * 1e9 * 0.98:
            raise RuntimeError(f'{fname} is {real/1e9:.2f} GB on disk but should be ~{expect:.2f} GB — the '
                               'download was cut short. Run this cell again; hf_hub_download resumes.')
        got[k] = path
        if not quiet:
            print(f'  ok {fname}: {fmt_size(real)}')
    if not quiet:
        print('weights ready: ' + ', '.join(os.path.basename(model_path(model, k)) for k in need))
    return got


# ------------------------------------------------------------------ 4. media helpers
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
VIDEO_EXTS = ('.mp4', '.mov', '.mkv', '.webm', '.avi', '.m4v', '.mpg', '.mpeg', '.wmv', '.flv', '.ts')
OUT_EXTS = ('.mp4', '.mkv', '.webm', '.png', '.jpg', '.jpeg', '.webp')


def media_kind(path):
    ext = os.path.splitext(str(path))[1].lower()
    if ext in VIDEO_EXTS:
        return 'video'
    if ext in IMAGE_EXTS:
        return 'image'
    return 'other'


def fmt_dur(seconds):
    if seconds is None:
        return '?'
    seconds = float(seconds)
    h, m, s = int(seconds // 3600), int(seconds % 3600 // 60), seconds % 60
    return (f'{h}:{m:02d}:' if h else '') + f'{m:02d}:{s:04.1f}'


def _fps_from_ratio(text, default=25.0):
    t = str(text or '')
    if '/' in t:
        a, b = t.split('/', 1)
        try:
            b = float(b)
            return float(a) / b if b else default
        except Exception:
            return default
    try:
        return float(t) or default
    except Exception:
        return default


_info_cache = {}


def probe_video(path):
    """{width,height,fps,n_frames,dur} for a video; {} on failure. Cached per file."""
    key = os.path.realpath(path)
    if key in _info_cache:
        return _info_cache[key]
    info = {}
    try:
        import imageio
        rdr = imageio.get_reader(path, 'ffmpeg')
        try:
            meta = rdr.get_meta_data() or {}
            w, h = (int(meta['size'][0]), int(meta['size'][1])) if meta.get('size') else (0, 0)
            if w <= 0 or h <= 0:
                im0 = next(iter(rdr))
                h, w = int(im0.shape[0]), int(im0.shape[1])
            n = meta.get('nframes')
            if not isinstance(n, int) or n <= 0 or n > 1e7:
                try:
                    n = int(rdr.count_frames())
                except Exception:
                    n = max(1, int(round((meta.get('duration') or 0) * float(meta.get('fps') or 25))))
            fps = _fps_from_ratio(meta.get('fps'))
            info = dict(width=w, height=h, fps=fps, n_frames=int(n or 0),
                        dur=(int(n or 0) / fps if fps else None))
        finally:
            try:
                rdr.close()
            except Exception:
                pass
    except Exception as exc:
        print(f'  ! could not read {os.path.basename(path)}: {exc}')
    _info_cache[key] = info
    return info


def image_size(path):
    """((w, h), frames, animated) for a still (or the first frame of an animated one)."""
    try:
        from PIL import Image
        with Image.open(path) as im:
            im.seek(0)
            fr = getattr(im, 'n_frames', 1)
            return (int(im.size[0]), int(im.size[1])), fr, fr > 1
    except Exception as exc:
        print(f'  ! could not read {os.path.basename(path)}: {exc}')
        return (0, 0), 0, False


def describe_media(path):
    """One honest line per file for the picker, including the trap cases."""
    kind = media_kind(path)
    if kind == 'image':
        (w, h), fr, anim = image_size(path)
        txt = f'{w}x{h}'
        if anim:
            txt += f' · ANIMATED ({fr} frames — only the first is used)'
        return txt, anim
    i = probe_video(path)
    if not i:
        return 'unreadable', True
    txt = (f'{i["width"]}x{i["height"]} · {fmt_dur(i["dur"])} · {i["n_frames"]} frames '
           f'@ {i["fps"]:.3g} fps')
    odd = (i['width'] % 2) or (i['height'] % 2)
    return (txt + ' · ODD SIZE (padded/cropped to even)', True) if odd else (txt, False)


def media_frames(path):
    if media_kind(path) == 'image':
        return 1
    return probe_video(path).get('n_frames', 0) or 0


# ------------------------------------------------------------------ 5. memory model + planner
# Bytes per OUTPUT pixel per frame held at once. Measured shape of the fork's own estimator
# (nodes.estimate_vram_usage) with its x4 safety factor folded in: the fp16 upscaled LQ canvas
# (6 B), the fp16/fp32 output canvas + tile weights (8 B with tiling, more without), the latent
# (2 B per latent frame, 1 per 4 frames) and the decode scratch. 78 B/px-frame when both tilers are
# off vs 60 with them on, and +8 in "full" because encode and decode scratch coexist there.
def fv_dense_expected(attention='sdpa', fallback='auto'):
    """Will this run end up in dense SDPA rather than the fork's sparse Triton kernel?

    Not a cosmetic difference: dense attention keeps the full L x L scores per tile and the model reads a
    wider window, so both the memory estimate and the resume fingerprint have to know. 'off' means the
    driver was told not to rebind anything, so whatever the picker says, the sparse path is what runs.
    """
    if str(fallback) == 'off':
        return False
    return str(attention) == 'sdpa' or str(fallback) == 'force'


def fv_bytes_per_outpx(mode='tiny', tiled_dit=True, tiled_vae=True, dense_attn=False):
    """Rough bytes per output pixel per frame, for the STARTING chunk only.

    Dense attention is charged 30% more: the score matrix is the one term that stops being a top-k slice
    of the blocks and becomes the whole (still tiled) sequence."""
    b = 60 if (tiled_dit and tiled_vae) else (78 if (tiled_dit or tiled_vae) else 96)
    b += 8 if mode == 'full' else 0
    return int(round(b * 1.3)) if dense_attn else b


def fv_out_dims(w, h, scale=2, resize=1.0):
    """What the file comes out as. Mirrors nodes.py exactly: the pre-shrink is int(H * resize_factor)
    (truncated, not rounded) and the scale is applied to THAT, so the reported size is the real one;
    the 128-padding happens inside and is cropped back off.

    The model only ever runs at 2x or 4x, so the factor of the FILE is `scale * resize`, and the shrink
    box in Config is what every other size is made with: 4x on a frame given at 0.75 is an honest 3x,
    at 56% of the pixels. There is deliberately no "3x" for the network itself — LQ_proj_in and the
    one-step DMD were trained to undo a fixed downsample, and a ratio they have never seen is the one
    case where the output stops being a re-render and starts being a guess.
    """
    sw, sh = (int(w * resize), int(h * resize)) if resize < 1.0 else (int(w), int(h))
    sw, sh = max(sw, 8), max(sh, 8)
    return sw * scale, sh * scale


def fv_pad_dims(w, h, scale=2, resize=1.0):
    ow, oh = fv_out_dims(w, h, scale, resize)
    return -(-ow // 128) * 128, -(-oh // 128) * 128


def fv_out_pixels(w, h, scale=2, resize=1.0):
    pw, ph = fv_pad_dims(w, h, scale, resize)
    return pw * ph


def free_vram_gb(idx=0):
    """(free, total) GiB on the card; the Kaggle T4 assumption when there is no CUDA."""
    try:
        import torch
        if torch.cuda.is_available():
            fr, tot = torch.cuda.mem_get_info(idx)
            return fr / 2**30, tot / 2**30
    except Exception:
        pass
    return 15.0, 16.0


def fv_plan(out_px, free, weights_gb, mode='tiny', tiled_dit=True, tiled_vae=True, hard_max=41,
            target=0.62, dense_attn=False):
    """A starting chunk size for the biggest frame in the queue.

    Deliberately NOT the largest that fits: this formula is written outside the GPU and can only model
    the canvases, not the DiT's activations, so on a real T4 it has been seen to promise ~2x more frames
    than the card allows. It therefore aims at ~62% of the budget and hands the rest to the driver,
    which measures the true per-frame cost on the first few frames and shrinks the chunk from that."""
    per = max(1, int(out_px)) * fv_bytes_per_outpx(mode, tiled_dit, tiled_vae, dense_attn)
    budget = max(0.5, free * 0.88 - weights_gb - 1.0)
    cost1 = per / 2**30
    if cost1 >= budget:
        return dict(chunk=0, cost_gb=cost1, budget_gb=budget, ok=False,
                    why=f'one frame of {out_px/1e6:.2f} Mpx output already needs {cost1:.2f} GB of the '
                        f'{budget:.2f} GB left for activations — lower "Scale", pre-shrink the input or '
                        'turn both tilers on')
    chunk = min(hard_max, max(1, int(budget * target // cost1)))
    return dict(chunk=int(chunk), cost_gb=chunk * cost1, budget_gb=budget, ok=True,
                why=f'{chunk} frame(s) x {out_px/1e6:.2f} Mpx x '
                    f'{fv_bytes_per_outpx(mode, tiled_dit, tiled_vae, dense_attn)} B'
                    f' = {chunk*cost1:.2f} GB of a {budget:.2f} GB budget'
                    + (' with dense attention charged in' if dense_attn else '')
                    + ' (the driver re-fits this by measurement)')


def fingerprint(**kw):
    return hashlib.sha1(json.dumps(kw, sort_keys=True).encode()).hexdigest()[:12]



try:
    import torch
    _n_gpu = torch.cuda.device_count()
    _names = [torch.cuda.get_device_name(i) for i in range(_n_gpu)] or ['none']
except Exception:
    _n_gpu, _names = 0, ['torch not importable']
print(f'\nGPUs visible: {_n_gpu} ({", ".join(_names)})')
if _n_gpu and 'Tesla T4' in _names[0]:
    print('T4s: ask for bf16 and you get fp16 — the driver maps "auto" to fp16 here, and fp16 is what '
          'these weights were released to run at on this card.')

# The fork's wan_video_dit imports triton at module level (its vendored sparse_sage kernels), even when
# attention runs through plain SDPA — so psutil/pyyaml/triton are checked here rather than blowing up in
# the Run cell. Nothing gets installed if it is already there.
for _mod, _pip in (('psutil', 'psutil'), ('yaml', 'pyyaml'), ('triton', 'triton')):
    try:
        __import__(_mod)
    except Exception:
        print(f'  {os.path.basename(_pip)} missing — installing it (nodes.py imports {_mod} at import time)…')
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', _pip], check=False)

_SMOKE_SRC = r"""
import inspect, json, os, sys
from unittest.mock import MagicMock
repo = sys.argv[1]
sys.path.insert(0, repo)
fp = MagicMock(); fp.models_dir = sys.argv[2]; fp.get_filename_list = MagicMock(return_value=[])
sys.modules.setdefault("folder_paths", fp); sys.modules.setdefault("comfy", MagicMock())
sys.modules.setdefault("comfy.utils", MagicMock())
os.chdir(repo)
import nodes
wd = getattr(nodes, "wan_video_dit", None)
try:
    src = inspect.getsource(wd.flash_attention) if wd else ""
except Exception:
    src = ""
try:
    import triton
    tri = getattr(triton, "__version__", "?")
except Exception as e:
    tri = "missing:" + type(e).__name__
capa = None
try:
    import torch
    if torch.cuda.is_available():
        capa = list(torch.cuda.get_device_capability()) + [torch.cuda.get_device_name(0)]
except Exception:
    pass
print("SMOKE " + json.dumps({"attn": getattr(wd, "ATTENTION_MODE", "?"),
      "kwargs": sorted(inspect.signature(nodes.flashvsr).parameters),
      "vaes": sorted(getattr(nodes, "VAE_MODEL_OPTIONS", [])),
      "pad": bool(getattr(nodes, "next_8n5", None)),
      "use_block": bool(getattr(wd, "USE_BLOCK_ATTN", True)),
      "block_ok": bool(getattr(wd, "BLOCK_ATTN_AVAILABLE", False)),
      "mask_routes": "attention_mask is not None" in src,
      "has_sparse": bool(wd and getattr(wd, "sparse_sageattn", None)),
      "triton": tri, "capa": capa}))
"""


def _smoke_import_nodes(repo=None, models_dir=None):
    """Import the fork exactly the way the driver will, in a throwaway process.

    Doing it here (not at run time) turns a broken clone or a missing kernel dependency into one clear
    message before the weights are downloaded, and it hands back the real kwarg list of flashvsr()."""
    repo = repo or REPO_DIR
    r = subprocess.run([sys.executable, '-c', _SMOKE_SRC, repo, models_dir or MODELS_DIR],
                       capture_output=True, text=True, timeout=900)
    line = next((l for l in (r.stdout or '').splitlines() if l.startswith('SMOKE ')), None)
    if not line:
        tail = (r.stderr or r.stdout or '')[-1400:]
        extra = ''
        if 'triton' in tail.lower():
            extra = ('\n  triton is imported by src/models/sparse_sage even for the SDPA path, so a torch '
                     'image without it breaks this at import time. It ships with the standard Kaggle '
                     'image — restart the session on that image, or install the triton version that '
                     'matches your torch (pip show torch, then the same major.minor of triton).')
        if 'sageattention' in tail.lower() or 'flash_attn' in tail.lower():
            extra += ('\n  sageattention / flash-attn are optional backends: this notebook asks for sdpa, '
                      'so you do not need them.')
        raise RuntimeError('The cloned FlashVSR repo cannot be imported, so nothing in the Run cell can work. '
                           'Import error tail:' + tail + extra)
    d = json.loads(line[6:])
    print('nodes.py imports cleanly · attention default:', d['attn'], '· flashvsr() kwargs:', len(d['kwargs']),
          '· VAEs offered:', ','.join(d['vaes']) or '?')
    if 'capa' in d:
        _atk = ('block_sparse_attn (mit-han-lab build)' if d.get('use_block') and d.get('block_ok')
                else 'sparse_sageattn (Triton int8)' if d.get('has_sparse') else 'whatever the build uses')
        _routes = ('selected by whether a mask was built, and SelfAttention always builds one'
                   if d.get('mask_routes') else 'selected by ATTENTION_MODE')
        print(f'  attention: this build calls {_atk} — {_routes}, so the "Attention" picker alone '
              'changes nothing there (treat it as decoration unless the driver rebinds it)')
        capa, tri = d.get('capa'), str(d.get('triton'))
        if capa and tuple(capa[:2]) < (8, 0):
            print(f'  this card is sm_{capa[0]}{capa[1]} ({capa[2]}) with triton {tri}: the int8 sparse '
                  'kernel this build insists on is not built for pre-Ampere, and triton>=3.3 breaks it on '
                  'Turing too. That is the "PassManager::run failed" crash, one second into the first pass '
                  '— a kernel failure, not a memory one. Keep "Enforce dense SDPA" ticked in Config: the '
                  'driver then drops the mask and runs real SDPA per tile, which is what has been seen to '
                  'finish on a Kaggle T4. The alternative in the fork\u2019s README (pip install "triton<3.3.0") '
                  'only helps if it also matches your torch, and this session does not need it.')
        else:
            try:
                _tmin = int(''.join(c for c in tri.split('.')[1] if c.isdigit()) or 0)
            except Exception:
                _tmin = 0
            if tri.startswith('3.') and _tmin >= 3:
                print('  note: triton ' + tri + ' — if a run ever dies with PassManager::run failed, that '
                      'is the sparse kernel, and the dense-SDPA tick in Config is the fix')
    missing = [k for k in ('resize_factor', 'chunk_size', 'mode', 'tiled_dit', 'tile_overlap')
               if k not in d['kwargs']]
    if missing:
        print('  note: this build has no ' + ', '.join(missing) + ' — the driver will not pass them')
    return d


_sm = _smoke_import_nodes()
FV_KWARGS = _sm['kwargs']
print('Install cell done:', time.strftime('%H:%M:%S'))


In [ ]:
#@title ##**Upload** { display-mode: "form" }
# title: Upload Media (from your PC)
# Colab's own file picker and nothing else: run the cell, the browser asks which files to send.
#
#     from google.colab import files
#     uploaded = files.upload()
#
# whatever you choose is written into /content/inputs, and EVERYTHING in that folder is what the Run cell
# upscales — images and videos through the same picker, the kind is taken from the extension. Running the cell
# again replaces the batch rather than adding to it; the reason is the last paragraph of this note.
#
# The queue table is the real contract with the GPU: kind, size, resolution, and for a video its fps and
# frame count. The blue number is how many model passes a file costs — one per frame, and 21 for a still,
# because a single picture is padded up to the pipeline's 8n+5 minimum anyway. A frame is seconds on a T4,
# not milliseconds, so read the total before starting a long run.
#
# Uploading goes through the browser: a 4 GB clip takes minutes and needs the tab awake.
#
# THIS CELL HAS NO FORM, AND EVERY RUN OF IT STARTS CLEAN: the previous uploads and the previous results are
# deleted first, then the browser asks what to send. That is deliberate — the queue and /content/outputs
# belong to one batch, and a leftover result for a file you no longer send would make "Resume" skip work it
# was never asked to skip. Nothing you have just picked can be lost by it: the bytes you choose are already
# in memory before anything is deleted. To add to the current batch instead, run a scratch cell with
# `_refresh()` (re-read the folder, delete nothing) or `rescan_w.value = True; _pick_and_save()`.
import os, json, re

if 'media_kind' not in globals() or 'probe_video' not in globals():
    raise RuntimeError('Run the Install cell first — it sets up ffmpeg, the model files and the media helpers.')

try:                                    # only exists inside a Colab runtime
    from google.colab import files as _cf
except Exception:
    _cf = None

# No form in this cell — running it asks for files, that is the whole interface. Two internal switches stay
# settable from a scratch cell, and they are the only two:
rescan_w = _V(False)      # True = do not open the browser, just re-read /content/inputs (nothing is deleted)
skip_w = _V('')           # names or parts of names to keep OUT of the queue; the file stays on disk, so
                          # emptying this brings it back — it exists so you do not have to re-upload 11 clips
                          # to leave one out

# The queue survives a kernel restart through /content/selected_media.json, which the Run cell also
# reads — so "I sent 12 clips and the runtime died" does not mean sending them again. Paths that are no
# longer on disk are dropped here, so a wiped /content shows as an empty queue, not as an error.
if 'SELECTED_MEDIA' not in globals():
    try:
        SELECTED_MEDIA = json.load(open(SELECTED_JSON))
    except Exception:
        SELECTED_MEDIA = []
SELECTED_MEDIA = [p for p in SELECTED_MEDIA if os.path.exists(p)]
_table_cache = {}


# ------------------------------------------------------------------ what is where
def _queued_inputs():
    if not os.path.isdir(INPUT_DIR):
        return []
    exts = tuple(IMAGE_EXTS + VIDEO_EXTS)
    return sorted(os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR) if f.lower().endswith(exts))


def _queued_outputs():
    if not os.path.isdir(OUTPUT_DIR):
        return []
    return sorted(os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(OUT_EXTS))


def _skip_patterns():
    return [p for p in re.split(r'[,;\s]+', str(skip_w.value or '').strip()) if p]


def _skipped(path):
    b = os.path.basename(path).lower()
    return any(p.lower() in b for p in _skip_patterns())


def _all_queued():
    """What the Run cell will process. Existence is part of it: a file deleted from the Files panel
    between two runs must leave the queue, not turn into a probe error on a missing path."""
    return sorted(p for p in (set(_queued_inputs()) | set(SELECTED_MEDIA))
                  if os.path.exists(p) and not _skipped(p))


def _passes(path):
    """Frames this file costs the model. A still is NOT free: every call is padded up to the pipeline's
    8n+5 minimum, so one picture is 21 frames of work here."""
    try:
        if media_kind(path) == 'image':
            return 21
        return int((probe_video(path) or {}).get('n_frames') or 0) or 1
    except Exception:
        return 0


def _table(path, note=''):
    """One queue row, as a line of text: name, size, what the probe found, what it costs."""
    if not os.path.exists(path):                       # deleted while the notebook was open: say so
        return f'    {os.path.basename(path):<34} gone from disk, dropped from the queue'
    try:
        key = (os.path.getsize(path), os.path.getmtime(path))
        if _table_cache.get(path) != key:
            _table_cache[path] = (key, describe_media(path), _passes(path))
        _, info, passes = _table_cache[path]
    except Exception as exc:
        info, passes = f'could not probe: {exc}', 0
    return (f'    {os.path.basename(path)[:34]:<34} {fmt_size(os.path.getsize(path)):>9} · {info}'
            f' · {passes} frame(s) of work{note}')


def _save_selection():
    try:
        json.dump(sorted(set(_all_queued())), open(SELECTED_JSON, 'w'), indent=1)
    except Exception as exc:
        print(f'could not write {SELECTED_JSON}: {exc!r}')


# ------------------------------------------------------------------ the queue, printed
def _refresh(msg=''):
    inputs, outputs = _queued_inputs(), _queued_outputs()
    extra = [p for p in SELECTED_MEDIA if p not in inputs]
    queue = _all_queued()
    hidden = len([p for p in (set(inputs) | set(extra)) if _skipped(p)])
    n_img = sum(1 for p in queue if media_kind(p) == 'image')
    n_vid = len(queue) - n_img
    passes = sum(_passes(p) for p in queue)
    total_mb = sum(os.path.getsize(p) for p in queue) / 1e6
    out_mb = sum(os.path.getsize(p) for p in outputs) / 1e6
    status_w.value = (
        (msg + '\n' if msg else '') +
        f'queue: {len(queue)} file(s) — {n_img} image(s), {n_vid} video(s), {total_mb:.1f} MB · '
        f'{passes} model pass(es)'
        + (f' ({hidden} hidden by skip_w)' if hidden else '') + '\n' +
        '\n'.join([_table(p) for p in inputs] + [_table(p, ' (kept from the previous pick)') for p in sorted(extra)])
        + (f'\noutputs already in {OUTPUT_DIR}: {len(outputs)} file(s), {out_mb:.1f} MB — at 3 s/frame '
           f'(a guess) this queue is ~{fmt_dur(passes * 3.0)}; the Run cell measures the real number on the '
           f'first file and re-estimates the rest' if outputs or queue else '')
        + ('' if queue else f'\nnothing queued: run this cell to pick files, or copy them into {INPUT_DIR}'))
    print(status_w.value)
    _save_selection()


# ------------------------------------------------------------------ the picker
def _sweep():
    """Empty the queue and the results, return (uploads, results) counts. Files only: the Run cell's working
    folders and its logs stay, because a log is how you find out what happened in a session that died."""
    n_in = n_out = 0
    for d, out_dir in ((INPUT_DIR, False), (OUTPUT_DIR, True)):
        for f in (sorted(os.listdir(d)) if os.path.isdir(d) else []):
            p = os.path.join(d, f)
            if not os.path.isfile(p):
                continue
            if out_dir and os.path.splitext(f)[1].lower() not in OUT_EXTS:
                continue
            try:
                os.remove(p)
                _table_cache.pop(p, None)
                if out_dir:
                    n_out += 1
                else:
                    n_in += 1
            except Exception as exc:
                print(f'could not remove {p}: {exc!r}')
    SELECTED_MEDIA[:] = [q for q in SELECTED_MEDIA if os.path.exists(q)]
    return n_in, n_out


def _pick_and_save(_=None):
    """The cell\u2019s own step: clear, ask the browser, write what arrives. Safe to call again \u2014 and a call
    from a scratch cell with nothing to upload deletes nothing (no files = nothing chosen = no sweep)."""
    got = {}
    if rescan_w.value:
        print('rescan_w is True: reading the folder, the browser is not asked and nothing is deleted.')
    elif _cf is None:
        print('note: google.colab.files only exists in a Colab runtime — reading the folder instead.')
    else:
        try:
            got = dict(_cf.upload() or {})
        except Exception as exc:
            print(f'the file picker reported {exc!r} — treating it as "nothing chosen".')
            got = {}
    cleared = _sweep() if got else (0, 0)      # only a real upload replaces the batch
    saved, names, skipped = 0, [], []
    os.makedirs(INPUT_DIR, exist_ok=True)
    for name, data in got.items():
        base = os.path.basename(name)
        if os.path.splitext(base)[1].lower() not in IMAGE_EXTS + VIDEO_EXTS:
            skipped.append(base)
            continue
        with open(os.path.join(INPUT_DIR, base), 'wb') as out:
            out.write(data.tobytes() if hasattr(data, 'tobytes') else bytes(data))
        saved += 1
        names.append(base)
    msg = (f'{saved} file(s) from the browser: ' + ', '.join(names[:6]) + ('…' if len(names) > 6 else '')
           + (f' — the previous batch was cleared first ({cleared[0]} upload(s), {cleared[1]} result(s))'
              if any(cleared) else '')
           if saved else 'nothing was chosen — the queue and the results were left exactly as they were')
    if skipped:
        msg += f' — skipped (not an image or video): {", ".join(skipped[:4])}'
    _refresh(msg)


def _clear_inputs(_=None):
    # the folder is listed BEFORE anything is deleted: re-listing it at the end would build rows for paths
    # that no longer exist, which is how this used to raise FileNotFoundError
    gone = set(_queued_inputs())
    n = 0
    for p in sorted(gone):
        try:
            os.remove(p)
            n += 1
        except Exception as exc:
            print(f'could not remove {p}: {exc!r}')
        _table_cache.pop(p, None)
    for p in [p for p in SELECTED_MEDIA if p in gone]:
        SELECTED_MEDIA.remove(p)
    _refresh(f'queue cleared: {n} uploaded file(s) removed, {len(SELECTED_MEDIA)} pick(s) kept')
    return n


def _clear_outputs(_=None):
    n = 0
    for p in _queued_outputs():
        try:
            os.remove(p)
            n += 1
        except Exception as exc:
            print(f'could not remove {p}: {exc!r}')
    _refresh(f'removed {n} output file(s)')
    return n


status_w = _V('')
_pick_and_save()


In [ ]:
#@title ##**Config** { display-mode: "form" }
# title: Config
# Four choices in this form; everything else runs at the tested default, printed below so you can see
# what was decided and why. The four that matter, in the order the pipeline uses them:
#
#   • upscale_by — 2 or 4, and only those: `LQ_proj_in` and the one-step DMD were trained to undo exactly
#     those two downsamplings. Any other factor is made the honest way — the shrink box below: 4 with 0.75
#     gives a real 3x file, at 56% of the pixels of 4x, and the model never leaves its trained ratio.
#   • shrink_source — how much of the picture the model is handed BEFORE the upscale. The file comes out at
#     `upscale_by × this`, so it is also the knob that makes a 12 MP photo or a 1080p clip fit one T4.
#   • weights — which decoder the result is built with. The default is the reference one (a real VAE), the
#     `tiny` options are ~1/3 faster and lighter but decode with TCDecoder, whose sub-pixel shuffle paints a
#     faint 1:2 brick mesh over flat areas; pick one for drafts, or when the card is the problem.
#   • vae — only the `full` options read this; `tiny` never loads a VAE at all.
#
# FIXED, ON PURPOSE (the Run cell prints what it changes): both tilers on, tile 256 with 24 overlap, chunk
# sized from the card and re-fitted by measurement, dense SDPA (the only attention a T4 has been seen to
# finish with), fp16 via auto, DiT freed before each decode, colour fix on, seed 0, video kept as the
# driver's x264 MP4 at CRF 16 with the source audio muxed back, stills as PNG, whole clip at the source
# fps, Resume on (a file made with these exact settings is not redone).
#
# The form is plain Python too: it sets variables, so a scratch cell can override anything (`chunk_w.value
# = 48`, `attn_w.value = 'sparse_sage_attention'`) and the Run cell will obey.
import os

if 'fv_plan' not in globals():
    raise RuntimeError('Run the Install cell first — it defines the model table, the helpers and the planner.')

# label -> (repo id, key files GB, mode, note). The first entry is the default, and it is also the
# configuration every memory number below is planned for.
MODEL_OPTIONS = {
    'FlashVSR v1.1 — full + Wan2.1 VAE (6.9 GB) — default, reference quality decoder': (
        'FlashVSR-v1.1', 6.95, 'full',
        'encodes + decodes with the real Wan2.1 VAE: no sub-pixel brick pattern on gradients, sharpest; '
        'the VAE sits on the card next to the DiT, so the chunk comes down a little'),
    'FlashVSR v1.1 — tiny + TCDecoder, fp16 (6.4 GB) — fastest, no VAE in VRAM': (
        'FlashVSR-v1.1', 6.44, 'tiny',
        'streaming DiT, decoder = TCDecoder: ~1/3 faster and no VAE resident, but the decoder\u2019s shuffle '
        'upsampling shows as a fine 1:2 mesh on flat areas; pick it for drafts or when the card is tight'),
    'FlashVSR v1.1 — tiny-long + TCDecoder (6.4 GB), lowest peak on long clips': (
        'FlashVSR-v1.1', 6.44, 'tiny-long',
        'same weights, per-latent-frame streaming: the peak is one latent frame instead of the whole chunk'),
    'FlashVSR v1.1 — full + Wan2.2 VAE (9.3 GB) — heaviest, most detail': (
        'FlashVSR-v1.1', 9.26, 'full',
        'Wan2.2 VAE is 2.8 GB of extra weights: keep the chunk small, expect it to be slow on a T4'),
    'FlashVSR v1.0 — tiny + TCDecoder (6.4 GB) — the original release': (
        'FlashVSR', 6.44, 'tiny',
        'slightly more detail than v1.1, more prone to flicker/over-sharpening; different LQ_proj weights'),
    'FlashVSR v1.0 — full + Wan2.1 VAE (6.9 GB)': ('FlashVSR', 6.95, 'full', 'original release, reference decoder'),
}
# what the "full" modes may decode with (0.02–2.82 GB); the file sizes are what the free disk pays
VAE_OPTIONS = [('Wan2.1', 0.508), ('Wan2.2', 2.819), ('LightVAE_W2.1', 0.032), ('TAE_W2.2', 0.023),
               ('LightTAE_HY1.5', 0.046)]
VAE_NAMES = [n for n, _gb in VAE_OPTIONS]
SCALE_OPTIONS = [('2\u00d7 — the tested, stable setting (recommended)', 2),
                 ('4\u00d7 — sharper, more invented, 4\u00d7 the pixels of 2\u00d7', 4)]
_DECODER_LABELS = {
    'full + Wan2.1 VAE — default, reference quality (recommended)':
        'FlashVSR v1.1 — full + Wan2.1 VAE (6.9 GB) — default, reference quality decoder',
    'tiny + TCDecoder — fastest, no VAE in VRAM, shows a 1:2 mesh on flats':
        'FlashVSR v1.1 — tiny + TCDecoder, fp16 (6.4 GB) — fastest, no VAE in VRAM',
    'tiny-long + TCDecoder — lowest peak on long clips':
        'FlashVSR v1.1 — tiny-long + TCDecoder (6.4 GB), lowest peak on long clips',
    'full + Wan2.2 VAE — heaviest, most detail':
        'FlashVSR v1.1 — full + Wan2.2 VAE (9.3 GB) — heaviest, most detail',
    'v1.0 tiny + TCDecoder — the original release':
        'FlashVSR v1.0 — tiny + TCDecoder (6.4 GB) — the original release',
    'v1.0 full + Wan2.1 VAE': 'FlashVSR v1.0 — full + Wan2.1 VAE (6.9 GB)',
}
_SCALES = {lab: v for lab, v in SCALE_OPTIONS}
_bad = [k for k in _DECODER_LABELS.values() if k not in MODEL_OPTIONS]
if _bad:
    raise RuntimeError(f'the form offers weights the notebook has no entry for: {_bad}')

upscale_by = "2× — the tested, stable setting (recommended)"  #@param ["2× — the tested, stable setting (recommended)", "4× — sharper, more invented, 4× the pixels of 2×"]
shrink_source = 1.0  #@param {type: "slider", min: 0.1, max: 1.0, step: 0.05}
weights = "full + Wan2.1 VAE — default, reference quality (recommended)"  #@param ["full + Wan2.1 VAE — default, reference quality (recommended)", "tiny + TCDecoder — fastest, no VAE in VRAM, shows a 1:2 mesh on flats", "tiny-long + TCDecoder — lowest peak on long clips", "full + Wan2.2 VAE — heaviest, most detail", "v1.0 tiny + TCDecoder — the original release", "v1.0 full + Wan2.1 VAE"]
vae = "Wan2.1"  #@param ["Wan2.1", "Wan2.2", "LightVAE_W2.1", "TAE_W2.2", "LightTAE_HY1.5"]
#   upscale_by — the factor the model renders at; shrink_source — the fraction of the picture it is given.
#   The file is the two multiplied: 4 and 0.75 is a real 3×, 4 and 0.5 is 2× made from a half-size frame.
#   vae — only the full decoders read it; in tiny / tiny-long the shipped TCDecoder decodes, so this field
#   is ignored there and those weights are never downloaded.
scale = _SCALES.get(upscale_by, 2)     # anything else than 2/4 falls back to 2 and the Run cell says so

# ------------------------------------------------------------------ the four choices + the fixed defaults
# The Run cell was written against widgets, so every setting keeps its widget name behind a plain holder.
model_w = _V(_DECODER_LABELS[weights], list(MODEL_OPTIONS))   # options = the real entries
scale_w = _V(int(scale), [v for _l, v in SCALE_OPTIONS])
resize_w = _V(round(float(shrink_source), 3))
vae_w = _V(vae if vae in VAE_NAMES else 'Wan2.1', VAE_NAMES)
prec_w = _V('auto')                                   # bf16 where the card has it, fp16 on a T4
attn_w = _V('sdpa')                                    # the only one a Turing card finishes with here
attn_fb_w = _V(True)                                   # and it rebinds to dense SDPA if the sparse kernel dies
offload_w = _V(False)                                  # ~12 GB of system RAM is no place for 7 GB of weights
unload_dit_w = _V(True)                                # free the DiT before each decode: the cheap way to fit
vram_headroom_w = _V(1500)                             # left untouched for fragmentation and cuDNN scratch
both_gpus_w = _V(False)                                # one GPU per Colab session; auto-detected if it gets two
split_w = _V(False)                                    # (and a second card never lowers what ONE file costs)
chunk_w = _V(0)                                        # 0 = the Run cell sizes it from the card, by measurement
tdit_w = _V(True)
tvae_w = _V(True)
tile_w = _V(256)
tov_w = _V(24)
sparse_w = _V(2.0)
kvr_w = _V(3.0)
lrange_w = _V(11)
colorfix_w = _V(True)
seed_w = _V(0)
start_w = _V(0.0)
end_w = _V(0.0)
maxframes_w = _V(0)
fps_w = _V('source')
fmt_w = _V('copy')
crf_w = _V(16)
preset_w = _V('fast')
img_w = _V('png')
imgq_w = _V(95)
audio_w = _V(True)
resume_w = _V(True)
cleanup_w = _V(True)
verbose_w = _V(False)
dry_w = _V(False)
est_info, vram_info, model_note, chunk_note = _V(''), _V(''), _V(''), _V('')


def _fit_lines():
    """Plain-text version of the fit panel: weights resident, what one frame costs, the chunk the planner
    would pick for the first file in the queue. Printed, not a widget — the number is the same either way."""
    label = model_w.value
    _rid, gb, mode, note = MODEL_OPTIONS[label]
    vae_gb = part_gb(vae_w.value) if mode == 'full' else 0.0      # tiny modes never load a VAE
    free, total = free_vram_gb()
    dense = fv_dense_expected(attn_w.value, 'auto' if attn_fb_w.value else 'off')
    rows = [('weights on the card', f'~{gb + vae_gb:.2f} GB ({gb:.2f} DiT+LQ_proj+TCDecoder'
             + (f' + {vae_gb:.2f} VAE' if mode == 'full' else ', no VAE in tiny modes') + ')'),
            ('card', f'{total:.1f} GB total, {free:.1f} GB free right now'),
            ('per output pixel-frame', f'{fv_bytes_per_outpx(mode, True, True, dense)} B/px'
             + (' (dense attention)' if dense else '')),
            ('output size', f'source × {scale} × {float(shrink_source):.2f} = '
             f'{scale * float(shrink_source):.2f}× of the original')]
    out_px = None
    try:
        for p in _all_queued():
            if media_kind(p) == 'video':
                i = probe_video(p)
                out_px = fv_out_pixels(i['width'], i['height'], scale_w.value, resize_w.value)
                rows[3] = ('output size', f'{fv_out_dims(i["width"], i["height"], scale_w.value, resize_w.value)[0]}'
                           f'×{fv_out_dims(i["width"], i["height"], scale_w.value, resize_w.value)[1]}'
                           f' = source × {scale} × {float(shrink_source):.2f}')
            else:
                (w0, h0), _f, _a = image_size(p)
                out_px = fv_out_pixels(w0, h0, scale_w.value, resize_w.value)
            break
    except Exception:
        out_px = None
    verdict = ''
    if out_px:
        plan = fv_plan(out_px, free, gb + vae_gb, mode=mode, tiled_dit=True, tiled_vae=True, dense_attn=dense)
        rows.append(('chunk', f'{plan["chunk"]} frame(s) per pass — {plan["cost_gb"]:.2f} GB of a '
                              f'{plan["budget_gb"]:.2f} GB budget' + ('' if plan['ok'] else '  ⚠ ' + plan['why'])))
        verdict = ('fits with room for the tile scratch' if plan['ok'] else
                   'tight: ' + plan['why'] + ' — the Run cell shrinks the chunk, then the tile, then the '
                                             'input, and leaves a file out with a note rather than dying at 90%')
    return rows, verdict, note


_rows, _verdict, _note = _fit_lines()
vram_info.value = '\n'.join(f'    {k:<20}{v}' for k, v in _rows) + (f'\n    {_verdict}' if _verdict else '')
model_note.value = _note
est_info.value = vram_info.value
chunk_note.value = ('the chunk is the whole memory story here: one pass holds chunk × output pixels, so the '
                    'Run cell starts from this estimate and then trusts what the card measured')
# ...and that is the whole cell. It prints nothing: the form is the interface, and the numbers above are
# computed and kept in vram_info.value / est_info.value for whoever wants them — read them from a scratch
# cell (`print(vram_info.value)`) rather than scrolling a panel after every change. The settings that are not
# offered here are fixed on purpose: both tilers on, tile 256/24, dense SDPA, fp16 via auto, colour fix on,
# seed 0, x264 MP4 at CRF 16 with the source audio, PNG for stills, source fps, Resume on.


In [ ]:
#@title ##**Run** { display-mode: "form" }
# title: Run
# The cell that does the work. Order of operations, all printed as it goes:
#   1. read the Config widgets, write the batch driver, make sure the selected weights are on disk
#   2. fix what cannot work (scale vs frame size, chunk vs free VRAM, tile bigger than the frame,
#      unsupported attention backend on this card) and say what was changed and why
#   3. probe every queued file, skip the ones already finished with these settings
#   4. split the leftovers over the card(s), one driver process per card, weights loaded ONCE per process
#   5. run, with per-file progress bars, a log file, and an OOM retry chain (chunk -> tiles -> pre-shrink)
#   6. finalize: audio muxed back from the source, result moved into /content/outputs together with
#      a settings fingerprint, so re-running the cell resumes instead of repeating
import os, re, sys, json, time, glob, shutil, subprocess, threading, ipywidgets as W
from IPython.display import display, HTML

# ---------------------------------------------------------------- what this cell shows, and what it only records
# Bars on the screen, text in a file. There is a lot worth reading once (the plan, the exact commands, what
# was fixed and why, the measurements, the report) and none of it is worth scrolling through before every
# run, so all of it is written to RUN_LOG while the cell itself leaves only the progress bars — plus one line
# when the run could not start, when it is a dry run, and when it ends. `_SHOW_LOG = True` in any cell brings
# the text back for the rest of the session; RUN_LOG holds it either way.
_SHOW_LOG = bool(globals().get('_SHOW_LOG', False))
RUN_LOG = f'{WORK}/run.log'
try:
    _run_log = open(RUN_LOG, 'a', buffering=1)
except Exception:
    _run_log = None                                   # a read-only /content would be a bad reason to fail
_real_print = globals().get('_real_print') or print    # the built-in, taken once and kept across re-runs
if globals().get('print') is not _real_print:           # a run that died mid-cell left the shadow behind, and
    globals()['print'] = _real_print                    # then the other cells would lose their output too


def _to_log(*a, sep=' ', end='\n'):
    """print() for this cell: into RUN_LOG, and onto the screen only while _SHOW_LOG is on."""
    text = sep.join(str(x) for x in a) + end
    if _run_log is not None:
        try:
            _run_log.write(text)
        except Exception:
            pass
    if _SHOW_LOG:
        _real_print(text, end='')


def _say(*a):
    """The few lines that reach the screen whatever _SHOW_LOG says: the run did not start, a dry run, and
    how it ended. One line each — the bars are the interface of this cell."""
    _real_print(*a)


print = _to_log        # noqa: A001 — everything printed below goes through _to_log until the last line
                      # (and the last line hands the built-in back; if the cell dies halfway, the next run of
                      # this cell takes it back — see the two lines above)

_MISSING = [n for n in ('REPO_DIR', 'DRIVER', 'MODEL_OPTIONS', '_all_queued') if n not in globals()]
if _MISSING:
    raise RuntimeError('Run the earlier cells first — missing: ' + ', '.join(_MISSING))

# The upscaler driver is kept inside this cell as text, so the notebook is one file and the script can
# never drift out of sync with the flags above. Raw string: what you see here is byte-for-byte the file.
DRIVER_SRC = r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
fvsr_batch.py - batch driver for FlashVSR, written by the notebook's Run cell.

Why this file exists: the FlashVSR repo (ComfyUI-FlashVSR_Stable) ships a CLI that takes ONE input
file and writes its result through cv2's MPEG-4 writer. For a notebook queue that is a bad trade -
the weights (~6.4 GB) would be reloaded for every file and every output would pass through a lossy
encoder on the way to the real one. So this driver imports the very same functions the node/CLI
call (``nodes.init_pipeline`` and ``nodes.flashvsr``) and loops the queue inside one process:
frames in through imageio (RGB float32, no colour subsampling, no re-encode), chunks through the
pipeline, PNG / high-quality H.264 out.

Everything FlashVSR-specific (attention backend, VAE choice, sparse-attention knobs, the OOM
fallback that turns on Tiled VAE then Tiled DiT) stays in ``nodes.py`` - this file only moves
pixels and reports progress in the log grammar the notebook parses:

    Processing file <i>/<n>
    Upscaling batch <c>/<m>
    Output saved to: <path>
    [<key>] completed successfully in <s>s
"""

import argparse
import json
import math
import os
import sys
import time
from unittest.mock import MagicMock

IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff')
VID_EXT = ('.mp4', '.mov', '.avi', '.mkv', '.m4v', '.webm', '.mpg', '.mpeg', '.wmv', '.flv', '.ts', '.m2ts')


def parse_args():
    p = argparse.ArgumentParser(description='Batch driver for FlashVSR', allow_abbrev=False)
    p.add_argument('--queue', required=True, help='folder holding the files to process, named <key><ext>')
    p.add_argument('--out', required=True, help='folder for the results')
    p.add_argument('--repo', required=True, help='clone of ComfyUI-FlashVSR_Stable (has nodes.py + posi_prompt.pth)')
    p.add_argument('--models_dir', default=None, help='where FlashVSR-v1.1/ lives (defaults to <repo>/models)')
    p.add_argument('--skip', default=None, help='optional JSON file: {"<key>": "reason"} to leave out')
    p.add_argument('--windows', default=None,
                   help='optional JSON file: {"<key>": {"lo": first source frame, "hi": one past the last, '
                        '"seed": n}} — the per-entry trim window that lets two processes take two halves of '
                        'one clip and stay exactly back-to-back (seconds would drift by rounding)')
    # pipeline (mirrors the node inputs 1:1)
    p.add_argument('--model', default='FlashVSR-v1.1', choices=['FlashVSR', 'FlashVSR-v1.1'])
    p.add_argument('--mode', default='tiny', choices=['tiny', 'tiny-long', 'full'])
    p.add_argument('--vae_model', default='LightVAE_W2.1',
                   choices=['Wan2.1', 'Wan2.2', 'LightVAE_W2.1', 'TAE_W2.2', 'LightTAE_HY1.5'])
    p.add_argument('--precision', default='auto', choices=['auto', 'bf16', 'fp16'])
    p.add_argument('--attention_mode', default='sdpa',
                   choices=['sdpa', 'flash_attention_2', 'sparse_sage_attention', 'block_sparse_attention'])
    p.add_argument('--device', default='cuda:0')
    p.add_argument('--scale', type=int, default=2, choices=[2, 4])
    p.add_argument('--resize_factor', type=float, default=1.0)
    p.add_argument('--frame_chunk_size', type=int, default=0,
                   help='CEILING for frames per pass (0 = start at --probe_chunk and let the measurement '
                        'decide). With --auto_chunk the driver re-sizes this from the card''s real peak.')
    p.add_argument('--auto_chunk', type=int, default=1,
                   help='1 = measure the real per-frame cost on this card (one small chunk, then '
                        'torch.cuda.max_memory_allocated) and cap the chunk so the rest of the clip fits')
    p.add_argument('--probe_chunk', type=int, default=5, help='frames in the first, measuring chunk')
    p.add_argument('--attn_fallback', default='auto', choices=['auto', 'off', 'force'],
                   help="this fork hard-codes USE_BLOCK_ATTN=False and hands every self-attention call to "
                        "the int8 block-sparse Triton kernel whatever --attention_mode says; on a Turing T4 "
                        "(and with triton>=3.3) that kernel dies in MLIR before memory is even touched. "
                        "auto = run real dense SDPA when sdpa was asked or when this card cannot do sparse; "
                        "force = always dense; off = keep this build's behaviour")
    p.add_argument('--vram_headroom_mb', type=int, default=1500,
                   help='memory left untouched for allocator fragmentation, cudnn workspaces, decode')
    p.add_argument('--tiled_vae', action='store_true')
    p.add_argument('--tiled_dit', action='store_true')
    p.add_argument('--tile_size', type=int, default=256)
    p.add_argument('--tile_overlap', type=int, default=24)
    p.add_argument('--unload_dit', action='store_true')
    p.add_argument('--sparse_ratio', type=float, default=2.0)
    p.add_argument('--kv_ratio', type=float, default=3.0)
    p.add_argument('--local_range', type=int, default=11, choices=[9, 11])
    p.add_argument('--color_fix', type=int, default=1)
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--keep_models_on_cpu', type=int, default=1)
    p.add_argument('--debug', action='store_true')
    # input trimming + output encoding
    p.add_argument('--start_sec', type=float, default=0.0)
    p.add_argument('--end_sec', type=float, default=0.0)
    p.add_argument('--max_frames', type=int, default=0)
    p.add_argument('--fps', type=float, default=0.0, help='0 = keep the source rate')
    p.add_argument('--vcodec', default='libx264')
    p.add_argument('--crf', type=int, default=14)
    p.add_argument('--preset', default='medium')
    p.add_argument('--pix_fmt', default='yuv420p')
    p.add_argument('--image_format', default='png', choices=['png', 'jpg', 'webp', 'same'])
    p.add_argument('--image_quality', type=int, default=95)
    p.add_argument('--probe', action='store_true', help='print what each file is (frames, size, fps) and stop')
    p.add_argument('--dry_run', action='store_true')
    return p.parse_args()


def list_items(queue_dir, skip_file, win_file=None):
    skip = {}
    if skip_file and os.path.exists(skip_file):
        try:
            skip = json.load(open(skip_file))
        except Exception:
            skip = {}
    wins = {}
    if win_file and os.path.exists(win_file):
        try:
            wins = json.load(open(win_file))
        except Exception:
            wins = {}
    out = []
    for name in sorted(os.listdir(queue_dir)):
        path = os.path.join(queue_dir, name)
        if not os.path.isfile(path) or name.startswith('.'):
            continue
        key, ext = os.path.splitext(name)
        ext = ext.lower()
        if key in skip:
            print(f'  - {key}: skipped ({skip[key]})', flush=True)
            continue
        if ext in VID_EXT:
            kind = 'video'
        elif ext in IMG_EXT:
            kind = 'image'
        else:
            continue
        it = dict(key=key, path=os.path.realpath(path), qpath=path, kind=kind, ext=ext)
        if key in wins and isinstance(wins[key], dict):
            it['_win'] = wins[key]
        out.append(it)
    return out


def probe(item):
    """(frames, (w, h), fps) for one file, using imageio's ffmpeg backend."""
    import imageio
    if item['kind'] == 'image':
        from PIL import Image
        with Image.open(item['path']) as im:
            w, h = im.size
        return 1, (w, h), 0.0
    rdr = imageio.get_reader(item['path'], 'ffmpeg')
    try:
        meta = {}
        try:
            meta = rdr.get_meta_data()
        except Exception:
            pass
        size = meta.get('size') or (0, 0)
        w, h = int(size[0]), int(size[1])
        if w <= 0 or h <= 0:
            for fr in rdr:
                h, w = fr.shape[0], fr.shape[1]
                break
        fps = float(meta.get('fps') or 0.0)
        n = meta.get('nframes')
        if not isinstance(n, int) or n <= 0:
            try:
                n = int(rdr.count_frames())
            except Exception:
                n = 0
        return max(1, int(n or 0)), (w, h), fps
    finally:
        try:
            rdr.close()
        except Exception:
            pass


def _window(item, args):
    """(first, last, frames_out, source_fps) for one queue entry: the trim window in SOURCE frame
    numbers plus how many frames come out after re-timing to --fps."""
    if '_p' not in item:
        item['_p'] = probe(item)
    n, _size, fps = item['_p']
    fps = fps or 25.0
    a = item.get('_win') or {}
    if 'lo' in a and 'hi' in a:
        lo, hi = int(a['lo']), int(a['hi'])          # exact frames: two halves meet without a gap
    else:
        lo = int(math.floor(args.start_sec * fps)) if args.start_sec > 0 else 0
        hi = int(math.floor(args.end_sec * fps)) if args.end_sec > 0 else n
    lo, hi = max(0, min(lo, n)), min(n, max(hi, lo + 1))
    cnt = hi - lo
    if args.max_frames > 0:
        cnt = min(cnt, args.max_frames)
    if args.fps > 0:
        cnt = max(1, int(round(cnt * args.fps / fps)))
    return lo, hi, cnt, fps


def _src_window(rdr, lo, hi):
    """Yield (index_in_window, frame) for the source frames inside [lo, hi)."""
    for idx, raw in enumerate(rdr):
        if idx < lo:
            continue
        if idx >= hi:
            return
        yield idx - lo, raw


def frames_of(item, args, want_total, lim=None):
    """Yield (N,H,W,C) torch float32 tensors in [0,1], one per chunk.

    `lim` is a mutable handle: {'chunk': frames per group, 'start': first OUTPUT frame to emit}. The
    caller changes 'chunk' after measuring the real cost, and resets 'start' after an OOM so a file is
    continued from where it stopped instead of being run again from frame 0.

    Re-timing is real resampling, not a container hint: frames are dropped when the rate goes down and
    repeated when it goes up, so 15 fps footage at 30 fps comes out as twice the frames and the same
    duration — a clip that only plays faster is a bug, not an fps change."""
    import imageio
    import numpy as np
    import torch
    if item['kind'] == 'image':
        # imageio's ffmpeg plugin refuses stills, and it would colour-subsample a JPEG anyway:
        # a still goes straight through Pillow, frame 0 only, RGB8.
        from PIL import Image
        with Image.open(item['path']) as im:
            try:
                im.seek(0)
            except Exception:
                pass
            a = np.asarray(im.convert('RGB')).astype(np.float32) / 255.0
        yield torch.from_numpy(a[None])
        return
    lim = lim if lim is not None else {'chunk': args.frame_chunk_size, 'start': 0}
    lo, hi, cnt, fps_src = _window(item, args)
    ratio = (fps_src / args.fps) if args.fps > 0 else 1.0      # source frames per output frame
    rdr = imageio.get_reader(item['path'], 'ffmpeg')
    try:
        gen = _src_window(rdr, lo, hi)
        state = next(gen, None)
        buf, out_i = [], max(0, int(lim.get('start', 0)))
        while out_i < cnt and state is not None:
            need = int(math.floor(out_i * ratio))
            while state[0] < need:
                nxt = next(gen, None)
                if nxt is None:
                    break
                state = nxt
            a = np.asarray(state[1])
            if a.ndim == 2:
                a = np.repeat(a[:, :, None], 3, axis=2)
            elif a.shape[-1] == 4:
                a = a[..., :3]
            buf.append(a.astype(np.float32) / 255.0)
            out_i += 1
            if len(buf) >= max(1, int(lim.get('chunk') or args.frame_chunk_size or 10 ** 9)):
                yield torch.from_numpy(np.stack(buf))
                buf = []
        if buf:
            yield torch.from_numpy(np.stack(buf))
    finally:
        try:
            rdr.close()
        except Exception:
            pass


class Writer:
    """H.264 through imageio (crf, not cv2's default MPEG-4), or PNG for stills."""

    def __init__(self, path, kind, fps, args, src_ext):
        import os
        self.path, self.kind, self.args = path, kind, args
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if kind == 'image':
            self.ext = {'same': src_ext, 'png': '.png', 'jpg': '.jpg', 'webp': '.webp'}[args.image_format]
            self.path = os.path.splitext(path)[0] + self.ext
            self.impl = None
        else:
            import imageio
            self.impl = imageio.get_writer(
                path, format='FFMPEG', mode='I', fps=fps or 25.0, codec=args.vcodec, macro_block_size=1,
                ffmpeg_params=['-crf', str(args.crf), '-preset', args.preset, '-pix_fmt', args.pix_fmt])
            self.ext = os.path.splitext(path)[1]

    def write(self, frames):
        """frames: torch (N,H,W,C) float in [0,1] (or numpy). A still writes frame 0 as an image."""
        import numpy as np
        t = frames.float().cpu() if hasattr(frames, 'float') else frames
        a = np.clip(np.asarray(t), 0.0, 1.0)
        a = (a * 255.0).round().astype(np.uint8)
        if self.kind == 'image':
            from PIL import Image
            im = Image.fromarray(a[0])
            low = self.path.lower()
            if low.endswith(('.jpg', '.jpeg')):
                im.convert('RGB').save(self.path, quality=self.args.image_quality)
            elif low.endswith('.webp'):
                im.save(self.path, quality=self.args.image_quality)
            else:
                im.save(self.path)
            return
        for i in range(a.shape[0]):
            self.impl.append_data(a[i])

    def close(self):
        if self.impl is not None:
            try:
                self.impl.close()
            except Exception:
                pass


# Signatures of "this card cannot build or load that kernel", as opposed to "it does not fit". Shrinking
# a chunk cannot fix any of them, so they must never start a memory retry.
_KERNEL_SIGNS = ('passmanager::run failed', 'no kernel image', 'ptxas', 'compilation error',
                 'unsupported gpu architecture', 'not supported on this architecture', 'misaligned address',
                 'cudaerrornotsupported', 'no viable alternative')


def _is_kernel_error(exc):
    t = f'{type(exc).__name__}: {exc}'.lower().replace('_', '').replace(' ', '')
    return any(k.replace(' ', '') in t for k in _KERNEL_SIGNS)


def dense_attention_on(mods, log, why):
    """Make "sdpa" mean sdpa in this fork, by dropping the block mask every self-attention call gets.

    Why a patch and not the flag: cli_main/nodes set wan_video_dit.ATTENTION_MODE and nothing ever reads it
    back. flash_attention() routes on `attention_mask is not None`, and SelfAttention.forward always builds
    that mask (generate_draft_block_mask_sage), which lands in sparse_sageattn() - a Triton int8 kernel the
    A100 was written for. On sm_75 that kernel fails at compile time, before any allocation, so the run
    dies with the same "exit 1" no chunk size would explain. Dropping the mask keeps the same q/k/v shapes
    and takes the SDPA path this build already uses for unmasked calls: full attention inside each tile
    instead of a top-k subset of it - slower, and the variant the one-step model approximates anyway.
    """
    orig = None
    for m in mods:
        try:
            f = getattr(m, 'flash_attention', None)
        except Exception:
            continue
        if callable(f) and not getattr(f, 'fvsr_dense', False):
            orig = orig or f
    if orig is None:
        return False

    def dense(q, k, v, num_heads, compatibility_mode=False, attention_mask=None, return_KV=False):
        if attention_mask is None:                     # cross-attention, and anything already dense
            return orig(q=q, k=k, v=v, num_heads=num_heads, compatibility_mode=compatibility_mode,
                        attention_mask=None, return_KV=return_KV)
        return orig(q=q, k=k, v=v, num_heads=num_heads, compatibility_mode=True, attention_mask=None)

    dense.fvsr_dense = True
    import sys
    hit = 0
    for m in list(sys.modules.values()) + [x for x in mods if x is not None]:
        try:                                                  # lazy module proxies can raise on getattr
            same = getattr(m, 'flash_attention', None) is orig
        except Exception:
            continue
        if same:
            try:
                m.flash_attention = dense
                hit += 1
            except Exception:
                pass
    log(f'  attention: dense SDPA on this card ({why}) - {hit} module(s) rebound, the block mask and the '
        'Triton sparse kernel are out of the way, so each tile now gets full attention')
    return True


def _card_capa(torch):
    try:
        if torch.cuda.is_available():
            return tuple(int(x) for x in torch.cuda.get_device_capability())
    except Exception:
        pass
    return None


def build_pipe(args, nodes_mod, torch, log=None):
    _log = log or (lambda m: print(m, flush=True))
    if args.precision == 'auto':
        dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) \
            else torch.float16
    else:
        dtype = torch.bfloat16 if args.precision == 'bf16' else torch.float16
    try:
        nodes_mod.wan_video_dit.ATTENTION_MODE = args.attention_mode
    except Exception as exc:                       # a build without that module attribute keeps its default
        print(f'note: could not set ATTENTION_MODE ({exc!r}) — using the backend this build defaults to',
              flush=True)
    args._attn = 'as shipped'
    if getattr(args, 'attn_fallback', 'auto') != 'off':
        capa = _card_capa(torch)
        asked = getattr(args, 'attn_fallback', 'auto') == 'force'
        if asked or args.attention_mode == 'sdpa' or (capa is not None and capa < (8, 0)):
            why = ('--attn_fallback force' if asked else
                   f'--attention_mode {args.attention_mode} was asked for, and this build ignores it'
                   if args.attention_mode == 'sdpa' else
                   f'compute capability {capa[0]}.{capa[1]} predates Ampere, and the block-sparse int8 '
                   'Triton kernel this build always uses is not built for it')
            if dense_attention_on([nodes_mod, getattr(nodes_mod, 'wan_video_dit', None)], _log, why):
                args._attn = 'dense sdpa'
    pipe = nodes_mod.init_pipeline(model=args.model, mode=args.mode, device=args.device,
                                   dtype=dtype, vae_model=args.vae_model)
    return pipe, dtype


def kw_supported(fn, names, log):
    """The fork keeps renaming node kwargs; only pass the ones this copy accepts."""
    import inspect
    try:
        ok = set(inspect.signature(fn).parameters)
    except (TypeError, ValueError):
        return set(names)
    keep, drop = set(), []
    for n in names:
        (keep.add(n) if n in ok else drop.append(n))
    if drop:
        log(f'note: this FlashVSR build has no {", ".join("--" + d for d in drop)}; not passed')
    return keep


class Vram:
    """Fit the chunk by MEASURING it, in the process that holds the model.

    A cost model written outside the GPU is a guess, and on FlashVSR a wrong guess is not a slow run
    but a dead one: the pipeline keeps the upscaled input, the latent, the DiT activations and the
    decoded chunk of the whole group at once, so what matters is what this card actually peaks at.
    So: run one small chunk, read the delta of torch.cuda.max_memory_allocated(), and derive the
    largest group that still leaves --vram_headroom_mb free. Nothing changes when there is no CUDA
    (the ceiling from the caller is then used as-is), so the driver stays usable for CPU debugging.
    """

    def __init__(self, torch, args, log):
        self.torch, self.args, self.log = torch, args, log
        self.on = bool(args.auto_chunk) and bool(torch.cuda.is_available())
        self.headroom = max(128, int(args.vram_headroom_mb)) * 1024 * 1024
        # bytes per input pixel per frame, NOT per frame: one queue mixes 256x192 clips with 4K stills,
        # and a single "MB per frame" figure would size every file by whichever happened to come first
        self.per_px = 0.0
        self.base = 0.0
        self.probes = 0

    def before(self):
        if not self.on:
            return
        t = self.torch
        try:
            t.cuda.synchronize()
        except Exception:
            pass
        try:
            self.base = float(t.cuda.memory_allocated())
        except Exception:
            self.base = 0.0
        try:
            t.cuda.reset_peak_memory_stats()
        except Exception:
            pass

    def after(self, n, px):
        """The chunk size to use from now on (None when measuring is off).

        px is the INPUT pixel count of one frame of this chunk: what carries over between files is
        bytes-per-pixel, so every file gets its own cap instead of inheriting the first file's size.
        """
        t = self.torch
        if not self.on or n <= 0 or px <= 0:
            return None
        try:
            peak = float(t.cuda.max_memory_allocated())
        except Exception:
            return None
        used = max(0.0, peak - self.base)
        if used <= 1 << 16:                     # nothing measured (fake/CPU torch): stay out of the way
            return None
        per_px = used / float(n) / float(px)
        self.probes += 1
        # the first pass allocates caches the later ones reuse, so keep the larger of the two readings
        self.per_px = max(self.per_px, per_px) if self.probes == 1 else (
            self.per_px * 0.4 + per_px * 0.6 if per_px < self.per_px else per_px)
        try:
            free, _tot = t.cuda.mem_get_info()
        except Exception:
            return None
        avail = max(1 << 24, float(free) - self.headroom)
        cap = max(1, int(avail // (self.per_px * float(px))))
        self.log(f'MEASURED probe={self.probes} frames={n} per_frame_mb={self.per_px * px / 1024**2:.1f} '
                 f'free_mb={float(free) / 1024**2:.0f} headroom_mb={self.headroom / 1024**2:.0f} '
                 f'cap_frames={cap} per_px_b={self.per_px:.1f}')
        return cap


def _is_oom(exc):
    return type(exc).__name__ == 'OutOfMemoryError' or 'out of memory' in str(exc).lower()


def process_one(pipe, NM, torch, it, args, want_kw, seed, log, vram, index):
    """One queue entry: decode -> measure -> feed the pipeline in chunks that fit, with in-process
    recovery. Returns (frames_written, stats) or raises if even a single frame does not fit."""
    seed = int((it.get('_win') or {}).get('seed', seed))   # a split clip shares one seed across its halves
    import numpy as np                                     # noqa: F401  (kept for parity with callers)
    kind = it['kind']
    n_fr, (w0, h0), fps = it.get('_p') or probe(it)
    want_total = _window(it, args)[2] if kind == 'video' else 1
    out_path = os.path.join(args.out, key_of(it) + ('.png' if kind == 'image' else '.mp4'))
    wr = Writer(out_path, kind, (args.fps or fps), args, it['ext'])
    ceiling = max(1, int(args.frame_chunk_size or 0))
    if vram.on:
        first = min(ceiling or 10 ** 9, max(1, int(args.probe_chunk)))
    else:
        first = ceiling or 10 ** 9
    lim = {'chunk': first, 'start': 0}
    n_chunks = max(1, -(-want_total // max(1, first)))
    kw0 = dict(scale=args.scale, color_fix=bool(args.color_fix), tiled_vae=args.tiled_vae,
               tiled_dit=args.tiled_dit, tile_size=args.tile_size, tile_overlap=args.tile_overlap,
               unload_dit=args.unload_dit, sparse_ratio=args.sparse_ratio, kv_ratio=args.kv_ratio,
               local_range=args.local_range, seed=seed,
               force_offload=bool(args.keep_models_on_cpu), enable_debug=args.debug,
               chunk_size=0, resize_factor=args.resize_factor, mode=args.mode)
    kw = {k: v for k, v in kw0.items() if k in want_kw}
    total, shown, ooms, t0, grew = 0, 0, 0, time.time(), False
    gen = frames_of(it, args, want_total, lim)
    try:
        while True:
            try:
                chunk = next(gen)
            except StopIteration:
                break
            n = int(chunk.shape[0])
            px = int(chunk.shape[2]) * int(chunk.shape[1])
            last_px = px
            shown += 1
            n_chunks = max(n_chunks, shown)
            log(f'Upscaling batch {shown}/{n_chunks}')
            vram.before()
            try:
                res = NM.flashvsr(pipe=pipe, frames=chunk, **kw)
            except Exception as exc:                        # noqa: BLE001
                if not _is_oom(exc):
                    # a kernel this card cannot build is not a size problem: switch the backend once and
                    # redo this pass, instead of shrinking pixels that were never the issue
                    if (_is_kernel_error(exc) and args.attn_fallback != 'off'
                            and dense_attention_on([NM, getattr(NM, 'wan_video_dit', None)], log,
                                                   'the attention kernel this build picked failed here: '
                                                   + str(exc)[:140])):
                        args._attn = 'dense sdpa (after a kernel failure)'
                        try:
                            torch.cuda.empty_cache()
                        except Exception:
                            pass
                        gen = frames_of(it, args, want_total, lim)
                        continue
                    raise
                ooms += 1
                if n <= 1:
                    raise RuntimeError(f'a single {int(chunk.shape[2])}x{int(chunk.shape[1])} frame still '
                                       f'exceeds this card ({exc}); lower the scale, pre-shrink the input, '
                                       f'or pick mode tiny') from exc
                lim['chunk'] = max(1, int(n * 0.6))        # 60% of what just failed, floored at one frame
                lim['start'] = total                        # resume right where the clip stopped
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                log(f'  OOM at {n} frames/chunk ({exc}); retrying this file from frame {total} with '
                    f'{lim["chunk"]}/pass')
                gen = frames_of(it, args, want_total, lim)
                continue
            del chunk
            wr.write(res)
            total += int(res.shape[0])
            del res
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            cap = vram.after(n, px)
            if cap:
                want = max(1, min(int(cap), ceiling) if ceiling else int(cap))
                cur = int(lim['chunk'])
                # Shrink whenever the measurement says the card cannot take it. Grow back only once and
                # only when it is clearly worth it, so a noisy reading cannot have the file re-decoding
                # itself on every chunk boundary.
                if want < cur or (not grew and want >= 2 * cur):
                    grew = grew or want > cur
                    log(f'  fitting chunk: {cur} → {want} from the measured '
                        f'{vram.per_px * px / 1024**2:.0f} MB/frame'
                        + (' (the ceiling asked for more than this card holds)' if ceiling and cap > ceiling
                           else ''))
                    lim['chunk'] = want
                    lim['start'] = total            # past the frames just written — never replay them
                    gen = frames_of(it, args, want_total, lim)
            if want_total and total >= want_total:
                break
        wr.close()
    except Exception:
        try:
            wr.close()
        except Exception:
            pass
        if os.path.isfile(out_path):
            try:
                os.remove(out_path)                     # a truncated file is worse than no file
            except OSError:
                pass
        raise
    if not os.path.exists(wr.path) or os.path.getsize(wr.path) == 0:
        raise RuntimeError('the driver produced an empty file')
    _sw, _sh = max(8, int(w0 * args.resize_factor)), max(8, int(h0 * args.resize_factor))
    secs = time.time() - t0
    log(f'[{it["key"]}] {total} frame(s) -> {_sw * args.scale}x{_sh * args.scale}'
        + ('' if args.resize_factor == 1.0 else f' (input pre-shrunk x{args.resize_factor:.2f})')
        + f', {shown} pass(es) at up to {lim["chunk"]} frames, {secs:.2f}s')
    log(f'Output saved to: {wr.path}')
    log(f'[{it["key"]}] completed successfully in {secs:.2f}s')
    return total, dict(seconds=round(secs, 2), passes=shown, chunk=int(lim['chunk']), ooms=ooms,
                       per_frame_mb=round(vram.per_px * max(1, last_px) / 1024**2, 1),
                       attn=getattr(args, '_attn', 'as shipped'), out=wr.path)


def key_of(it):
    return it['key']


def main():
    args = parse_args()
    t_boot = time.time()
    items = list_items(args.queue, args.skip, getattr(args, 'windows', None))
    if not items:
        print('Nothing to do: the queue folder is empty.', flush=True)
        return 0
    if args.probe:
        out = {}
        for it in items:
            n, (w, h), fps = probe(it)
            out[it['key']] = dict(kind=it['kind'], frames=n, size=[w, h], fps=round(fps, 6))
            print(f"{it['key']}: {it['kind']} {w}x{h} {n} frame(s) {fps:.3f} fps", flush=True)
        print('PROBE ' + json.dumps(out), flush=True)
        return 0

    # --- make the fork importable without ComfyUI (same mocks its own cli_main.py uses)
    sys.path.insert(0, args.repo)
    models_dir = args.models_dir or os.path.join(args.repo, 'models')
    fp = MagicMock()
    fp.models_dir = models_dir
    fp.get_filename_list = MagicMock(return_value=[])
    sys.modules.setdefault('folder_paths', fp)
    sys.modules.setdefault('comfy', MagicMock())
    sys.modules.setdefault('comfy.utils', MagicMock())
    os.chdir(args.repo)

    import torch

    def log(msg):
        print(msg, flush=True)

    import nodes as NM                                    # noqa: N806  (the fork's node module)
    if torch.cuda.is_available() and args.device.startswith('cuda'):
        try:
            torch.cuda.set_device(args.device)
        except Exception as exc:
            log(f'note: set_device({args.device}) failed: {exc!r}')
    if args.dry_run:
        log(f'[driver] would load {args.model} / {args.mode} / {args.vae_model} on {args.device} '
            f'({args.precision}) and process {len(items)} file(s): ' + ', '.join(i['key'] for i in items))
        return 0
    log(f'[driver] loading the pipeline (this is the only time the {args.model} weights are read)…')
    pipe, dtype = build_pipe(args, NM, torch, log)
    fr_tot = 0.0
    if torch.cuda.is_available():
        try:
            fr_tot = torch.cuda.mem_get_info()[0] / 1024**3
        except Exception:
            pass
    log(f'[driver] ready in {time.time() - t_boot:.1f}s · free after load: {fr_tot:.1f} GB · '
        f'{"chunking is measured on this card" if Vram(torch, args, log).on else "chunk ceiling taken as given"}')

    want_kw = kw_supported(NM.flashvsr, ['scale', 'color_fix', 'tiled_vae', 'tiled_dit', 'tile_size',
                                         'tile_overlap', 'unload_dit', 'sparse_ratio', 'kv_ratio',
                                         'local_range', 'seed', 'force_offload', 'enable_debug',
                                         'chunk_size', 'resize_factor', 'mode'], log)
    vram = Vram(torch, args, log)
    stats, done, oom_keys, bad_keys = {}, 0, [], []
    for i, it in enumerate(items, 1):
        log(f'\nProcessing file {i}/{len(items)}')
        try:
            total, st = process_one(pipe, NM, torch, it, args, want_kw, args.seed + i - 1, log, vram, i)
            stats[it['key']] = st
            done += 1
        except Exception as exc:                            # noqa: BLE001 - one bad file must not kill the queue
            import traceback
            log(f'ERROR in {it["key"]}: {exc!r}')
            traceback.print_exc()
            if _is_oom(exc):
                oom_keys.append(it['key'])
                log(f'OOM on {it["key"]} even after in-process shrinking — leaving it to the notebook, '
                    'continuing with the queue')
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
            else:
                bad_keys.append(it['key'])
    if stats:
        log('STATS ' + json.dumps(stats))
    log(f'\nDriver finished: {done}/{len(items)} file(s) in {time.time() - t_boot:.1f}s'
        + (f', OOM on {len(oom_keys)} ({", ".join(oom_keys[:4])})' if oom_keys else '')
        + (f', failed {len(bad_keys)}' if bad_keys else ''))
    return 86 if oom_keys else (0 if done else 1)


if __name__ == '__main__':
    sys.exit(main())'''

# ------------------------------------------------------------------ 1. what was asked for
_RID, _GB, _MODE, _NOTE = MODEL_OPTIONS[model_w.value]
S = dict(model=_RID, mode=_MODE, vae=vae_w.value, precision=prec_w.value, attention=attn_w.value,
         scale=int(scale_w.value), resize=round(float(resize_w.value), 3),
         chunk=int(chunk_w.value), tiled_dit=bool(tdit_w.value), tiled_vae=bool(tvae_w.value),
         tile=int(tile_w.value), tile_overlap=int(tov_w.value), unload_dit=bool(unload_dit_w.value),
         offload=bool(offload_w.value), sparse_ratio=float(sparse_w.value), kv_ratio=float(kvr_w.value),
         local_range=int(lrange_w.value), color_fix=bool(colorfix_w.value), seed=int(seed_w.value),
         attn_fallback=('auto' if attn_fb_w.value else 'off'),
         start_sec=float(start_w.value), end_sec=float(end_w.value), max_frames=int(maxframes_w.value),
         fps=(0.0 if fps_w.value == 'source' else float(fps_w.value)), vfmt=fmt_w.value,
         crf=int(crf_w.value), preset=preset_w.value, image_format=img_w.value,
         image_quality=int(imgq_w.value), audio=bool(audio_w.value), both_gpus=bool(both_gpus_w.value),
         headroom=int(vram_headroom_w.value), split=bool(split_w.value),
         resume=bool(resume_w.value), cleanup=bool(cleanup_w.value), debug=bool(verbose_w.value),
         dry_run=bool(dry_w.value))
FIX = []                                    # what was changed, and why — printed in the report
_RESIZE_ASKED = S['resize']


def _fix(what, why):
    FIX.append((what, why))
    print(f'  ! {what} — {why}')


# the driver is part of the notebook: rewriting it every run keeps the file and this cell in step
with open(DRIVER, 'w', encoding='utf-8') as f:
    f.write(DRIVER_SRC)
print(f'driver written to {DRIVER} ({len(DRIVER_SRC.splitlines())} lines)')

if S['attention'] != 'sdpa':
    _want, _ok = S['attention'], False
    for _m in ('sageattention', 'flash_attn', 'block_sparse_attn'):
        try:
            __import__(_m)
            _ok = True
            break
        except Exception:
            pass
    if not _ok:
        S['attention'] = 'sdpa'
        _fix(f'attention “{_want}” → sdpa',
             'its package is not installed (this notebook builds no CUDA kernels) — sdpa is slower than the '
             'paper’s A100 numbers and is the only backend that works on a T4 out of the box')
if S['precision'] == 'bf16':
    try:
        import torch
        if not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
            S['precision'] = 'fp16'
            _fix('precision bf16 → fp16', 'this GPU has no bf16 unit (Tesla T4 is Turing: fp16 only)')
    except Exception:
        pass
if S['scale'] not in (2, 4):
    _sc, S['scale'] = S['scale'], 2
    _fix(f'scale {_sc} → 2', 'the network only knows 2x and 4x — LQ_proj_in and the one-step DMD were '
                             'trained on those two downsamplings. For another size keep 4x and shrink the '
                             'source in Config instead: 4x with 0.75 is a real 3x file')
# how the two boxes multiply, in the words the user picked: "4x" is what the model does, and this says
# what the FILE is, whenever a shrink is involved
_sz = '' if S['resize'] >= 1.0 else f' on a source at {S["resize"]:.2f}×'
if S['tile_overlap'] < 16 and S['tiled_dit']:
    _tv, S['tile_overlap'] = S['tile_overlap'], 16
    _fix(f'tile overlap {_tv} → 16', 'below ~16 px the tiles leave visible seams on gradients and skies')

# ------------------------------------------------------------------ 2. weights on disk
if not all(model_present(S['model'], k) for k in ('dit', 'lq', 'tcd')):
    print('the weights are not here yet — downloading them now (the slowest part of a first run)')
    fetch_weights(S['model'], S['mode'], S['vae'])
elif S['mode'] == 'full' and not model_present(S['model'], S['vae']):
    print(f'mode “full” needs the {S["vae"]} decoder file, fetching it')
    fetch_weights(S['model'], 'full', S['vae'])
else:
    print(f'weights present: {S["model"]} / {S["mode"]}'
          + ('' if S['mode'] != 'full' else f' + VAE {S["vae"]}')
          + ' (tiny modes decode with the shipped TCDecoder, so no VAE file is needed)')
_w_gb = _GB + (part_gb(S['vae']) if S['mode'] == 'full' else 0.0)

# ------------------------------------------------------------------ 3. queue, with a reason for every skip
STATE = {}
if os.path.isfile(STATE_PATH):
    try:
        STATE = json.load(open(STATE_PATH))
    except Exception:
        STATE = {}
# What counts as "the same result": only the knobs that change PIXELS. Chunk size, tiling, offload,
# the memory headroom and the GPU count change speed or the wrapper, not the picture — and the planner
# recomputes the chunk from the card's free memory on every run, so leaving it in here would make
# "Resume" useless (a different free-VRAM reading would redo the whole queue every time).
FP_IGNORE = ('chunk', 'tile', 'tile_overlap', 'tiled_dit', 'tiled_vae', 'offload', 'unload_dit',
             'cost_px', 'headroom', 'dense_attn', 'both_gpus', 'split', 'resume', 'cleanup', 'debug',
             'dry_run')
FP = fingerprint(**{k: v for k, v in S.items() if k not in FP_IGNORE})
print(f'settings fingerprint {FP} · Resume ignores {", ".join(FP_IGNORE)}')

INFO, items = {}, []
_taken = set()
for p in _all_queued():
    key = re.sub(r'[^A-Za-z0-9_.-]+', '_', os.path.splitext(os.path.basename(p))[0])[:60] or 'file'
    if key in _taken:
        # the queue folder and the state file are both keyed by this name, so two "clip.mp4" in
        # different folders would silently overwrite each other — disambiguate instead.
        _n = 2
        while f'{key[:56]}_{_n}' in _taken:
            _n += 1
        print(f'  note: two files share the name “{key}” — the later one is handled as “{key[:56]}_{_n}”')
        key = f'{key[:56]}_{_n}'
    _taken.add(key)
    kind = media_kind(p)
    if kind == 'video':
        i = probe_video(p)
        if not i or i['width'] <= 0 or i['height'] <= 0:
            print(f'  - {key}: skipped (the video cannot be read)')
            continue
        w0, h0, fps_src = i['width'], i['height'], i['fps'] or 25.0
        n = int(i['n_frames'] or 0)
        if S['end_sec'] > S['start_sec'] >= 0:
            n = int((S['end_sec'] - S['start_sec']) * fps_src)
        elif S['end_sec'] > 0:
            n = int(S['end_sec'] * fps_src)
        if S['max_frames'] > 0:
            n = min(n or S['max_frames'], S['max_frames'])
        if i['n_frames']:
            n = min(n, i['n_frames']) if n else i['n_frames']
        n_src = max(0, int(n or 0))                    # SOURCE frames in the window, before re-timing
        if S['fps'] > 0:
            n = max(1, int(round(n * S['fps'] / fps_src)))
        frames = max(int(n or 0), 1)
    else:
        (w0, h0), _fr, _anim = image_size(p)
        if w0 <= 0 or h0 <= 0:
            print(f'  - {key}: skipped (the image cannot be read)')
            continue
        frames, fps_src = 21, 0.0        # a still is padded up to the 8n+5 minimum inside the pipeline
    st = STATE.get(key) or {}
    if S['resume'] and st.get('fp') == FP and st.get('out') and \
            os.path.isfile(os.path.join(OUTPUT_DIR, st['out'])):
        print(f'  - {key}: already done with these settings → kept as {st["out"]}')
        continue
    if (w0 % 2) or (h0 % 2):
        print(f'  note: {key} is {w0}x{h0} (odd) — the pipeline pads to a multiple of 128 and crops back, '
              'so it is fine, just slightly slower than an even source')
    INFO[key] = dict(path=p, kind=kind, w=w0, h=h0, frames=frames, fps=fps_src,
                     px=fv_out_pixels(w0, h0, S['scale'], S['resize']),
                     out=fv_out_dims(w0, h0, S['scale'], S['resize']),
                     n_src=int(i['n_frames'] or 0) if kind == 'video' else 1,
                     src_lo=int(round(S['start_sec'] * fps_src)) if (kind == 'video' and S['start_sec'] > 0)
                     else 0,
                     src_frames=n_src if kind == 'video' else 1)
    items.append(key)
RESULTS = []
if not items:
    _say(f'Nothing ran: every queued file is already finished with these settings or unreadable — '
         f'the results are in {OUTPUT_DIR}. Change a setting, or untick "Resume" (Advanced) to redo them.')
    sys.exit(0)

# ------------------------------------------------------------------ 4. the "will it fit" part
free, total = free_vram_gb(0)
big_px = max(INFO[k]['px'] for k in items)
S['dense_attn'] = fv_dense_expected(S['attention'], S['attn_fallback'])
S['cost_px'] = fv_bytes_per_outpx(S['mode'], S['tiled_dit'], S['tiled_vae'], S['dense_attn'])
plan = fv_plan(big_px, free, _w_gb, mode=S['mode'], tiled_dit=S['tiled_dit'], tiled_vae=S['tiled_vae'],
               dense_attn=S['dense_attn'])
print(f'\ncard reading: {free:.1f} GB free of {total:.1f} GB on cuda:0 · weights {_w_gb:.2f} GB · '
      f'biggest output frame {big_px/1e6:.2f} Mpx at {S["cost_px"]} B/px-frame')
print(f'model runs at {S["scale"]}\u00d7'
      + ('' if S['resize'] >= 1.0 else
         f', on a source shrunk to {S["resize"]:.2f}\u00d7 first — so the file comes out at '
         f'{S["scale"] * S["resize"]:.2f}\u00d7 of the original, which is the honest way to ask for 3\u00d7 '
         'or 2.5\u00d7 here'))
if S['chunk'] <= 0:
    S['chunk'] = plan['chunk']
    if S['chunk'] <= 0:
        for _rf in (0.75, 0.5, 0.35):
            _px = int(big_px * _rf * _rf)
            _pl = fv_plan(_px, free, _w_gb, mode=S['mode'], tiled_dit=True, tiled_vae=True,
                          dense_attn=S['dense_attn'])
            if _pl['chunk'] >= 1:
                S['resize'] = round(S['resize'] * _rf, 3)
                S['tiled_dit'] = S['tiled_vae'] = True
                S['cost_px'] = fv_bytes_per_outpx(S['mode'], True, True)
                plan = _pl
                S['chunk'] = _pl['chunk']
                _fix(f'pre-shrink input → {S["resize"]}, both tilers on',
                     f'at {big_px/1e6:.2f} Mpx output even ONE frame needs {plan["cost_gb"]:.2f} GB against a '
                     f'{plan["budget_gb"]:.2f} GB budget. The model still ran at '
                     f'{S["scale"]}\u00d7 — it is the frame it was given that shrank, so the file comes out at '
                     f'{S["scale"] * S["resize"]:.2f}\u00d7 of the original. That number is yours to choose: the '
                     'shrink box in Config, with 4x, is how a real 3x is made here')
                break
        else:
            raise RuntimeError(f'Nothing fits: a single {big_px/1e6:.2f} Mpx output frame needs more than the '
                               f'{free:.1f} GB free on the card even tiled. Lower "Scale" to 2 or shrink the '
                               'input yourself.')
    print(f'chunk sized from the card: {S["chunk"]} frame(s) per pass — {plan["why"]}')
    if S['chunk'] < 21 and any(INFO[k]['kind'] == 'image' for k in items):
        print('  note: a still costs 21 padded frames whatever the chunk is, so pictures are the tight case '
              'here; the retry chain shrinks them if the card disagrees')
elif S['chunk'] > plan['chunk'] and plan['chunk'] >= 1:
    _c, S['chunk'] = S['chunk'], plan['chunk']
    _fix(f'frames per chunk {_c} → {S["chunk"]}',
         f'{_c} frames at {big_px/1e6:.2f} Mpx is {_c * plan["cost_gb"] / max(1, plan["chunk"]):.2f} GB against '
         f'a {plan["budget_gb"]:.2f} GB budget')
    print(f'chunk capped from the card: {S["chunk"]} frame(s) per pass — {plan["why"]}')
# tiled DiT tiles the frame BEFORE upscaling (nodes.process_chunk uses the input shape), so the number to
# judge a tile against is the input's longest edge after pre-shrink — not the output size.
_in_long = max(max(INFO[k]['w'], INFO[k]['h']) * S['resize'] for k in items)
if S['tile'] >= _in_long:
    _t = S['tile']
    S['tile'] = max(128, int(_in_long // 2))
    _fix(f'tile {_t} → {S["tile"]}', f'the longest input edge here is {int(_in_long)} px, so a {_t} px tile '
         'splits nothing and costs a full extra canvas per tile instead of one')
if S['mode'] != 'full':
    print(f'  note: mode “{S["mode"]}” decodes with TCDecoder — “VAE” is ignored and its weights are not '
          'downloaded (that is where the 0.5-2.8 GB difference in the table comes from)')
print(f'plan: chunk ceiling {S["chunk"]} (the driver measures the real cost on the first {min(5, S["chunk"])} '
      f'frames and shrinks it if this card says so) · tile {S["tile"]}/{S["tile_overlap"]} · '
      + ('Tiled DiT ' if S['tiled_dit'] else '') + ('Tiled VAE ' if S['tiled_vae'] else '')
      + ('· DiT freed before decode ' if S['unload_dit'] else '')
      + f'· budget {plan["budget_gb"]:.2f} GB, planned {S["chunk"] * plan["cost_gb"] / max(1, plan["chunk"]):.2f} GB')

def _recalc_dims(note=''):
    """The plan may have shrunk the input; re-derive what each file will actually come out as, so the
    report and the state file never promise a size the driver did not produce."""
    for k in items:
        INFO[k]['px'] = fv_out_pixels(INFO[k]['w'], INFO[k]['h'], S['scale'], S['resize'])
        INFO[k]['out'] = fv_out_dims(INFO[k]['w'], INFO[k]['h'], S['scale'], S['resize'])
    if note:
        print('  ' + note)


_recalc_dims('sizes recomputed to match the plan' if S['resize'] != _RESIZE_ASKED else '')

def _row(k, out_path):
    """One result row, shaped the way the Visualization and Download cells expect: `out` is the full path
    (so a viewer can open it), `out_name` is what the report prints, and the sizes are the ones the
    plan promised after any auto-shrink — not the ones the Config tab showed before it."""
    i = INFO[k]
    H = globals().get('HIST') or {}
    sz = os.path.getsize(out_path) if out_path and os.path.exists(out_path) else 0
    r = dict(key=k, kind=i['kind'], src=i['path'], src_name=os.path.basename(i['path']),
                out=out_path or '', out_name=os.path.basename(out_path) if out_path else '-',
                w=i['out'][0], h=i['out'][1], src_size=[i['h'], i['w']], out_size=list(i['out']),
                frames=i['frames'], batches=max(1, -(-i['frames'] // max(1, S['chunk']))),
                fps=(S['fps'] or i['fps']), clip_seconds=(i['frames'] / i['fps']) if (i['kind'] == 'video'
                and i['fps']) else 0.0, seconds=H.get(k, {}).get('seconds'),
                peak_gb=H.get(k, {}).get('peak_gb'), size=sz, out_mb=sz / 1e6)
    if k in SPLIT:                                     # the clip ran as halves on two cards: total them
        h = [H.get(pk, {}) for pk in SPLIT[k]]
        secs = [x.get('seconds') for x in h if x.get('seconds')]
        pk = [x.get('peak_gb') for x in h if x.get('peak_gb')]
        r['seconds'] = round(sum(secs), 2) if secs else None
        r['peak_gb'] = max(pk) if pk else None
    return r


# ------------------------------------------------------------------ 5. jobs: one driver process per card
SPLIT, PARTWIN = {}, {}      # parent → its parts, in order;  part → {lo, hi, seed} for the driver


def _can_split(k):
    """A video long enough that both halves get at least one whole chunk. A trim window is fine — the
    part windows are built inside it — but --max_frames is not: the driver applies it per entry, so two
    parts would each be allowed the full limit."""
    i = INFO[k]
    return (i['kind'] == 'video' and i['frames'] >= 2 * max(1, int(S['chunk']))
            and int(i.get('src_frames') or 0) >= 2 and not S['max_frames'])



def _split_into(k, nparts=2):
    """Cut one clip into parts that meet exactly, so the second T4 has work while the first is busy.

    The cut lands on a multiple of the chunk on purpose: that is a place where this pipeline restarts
    its streaming cache anyway, so a split adds a boundary the run already had, not a new kind of seam.
    Every part also gets the parent's seed, so the invented detail does not change at the join, and the
    windows are passed in SOURCE FRAME NUMBERS rather than seconds — seconds round, and a rounding gap
    between two halves is a dropped frame at the seam."""
    i = INFO[k]
    fps_src = i['fps'] or 25.0
    eff = S['fps'] or fps_src
    ratio = fps_src / eff                               # source frames per output frame
    F = int(i['frames'])
    lo_s = int(i.get('src_lo') or 0)
    hi_s = lo_s + int(i.get('src_frames') or 0)
    if hi_s <= lo_s + 1:
        return None
    ch = max(1, int(S['chunk']))
    step = max(ch, int(round(max(ch, -(-F // nparts)) / ch)) * ch)
    cuts, cur = [], 0
    while cur < F:
        nxt = min(F, cur + step)
        if 0 < F - nxt < ch:
            nxt = F                                     # never leave a runt of a final part
        cuts.append((cur, nxt))
        cur = nxt
    if len(cuts) < 2:
        return None
    seed = S['seed'] + max(0, items.index(k)) if k in items else S['seed']
    parts = []
    for a, (c0, c1) in enumerate(cuts):
        key = f'{k}__p{a}'
        s0 = lo_s + int(round(c0 * ratio))
        s1 = hi_s if c1 >= F else lo_s + int(round(c1 * ratio))
        s1 = max(s1, s0 + 1)
        PARTWIN[key] = dict(lo=s0, hi=s1, seed=seed)
        INFO[key] = dict(i, frames=max(1, c1 - c0), part_of=k, part=a)
        parts.append(key)
    SPLIT[k] = parts
    return parts


def _mk_jobs(keys):
    """Balance by cost (frames x output pixels), biggest first, onto the freest card — and if that leaves
    a card with nothing to do while a clip is long enough to cut, cut it.

    Splitting costs a second copy of the weights on the second card, so it is worth it exactly when one
    file would otherwise run alone: one clip on two T4s is close to 2x wall clock, which is the case
    where 'use both cards' used to quietly do nothing."""
    cost = {k: max(1, INFO[k]['frames']) * INFO[k]['px'] for k in keys}
    n_dev = 1
    try:
        import torch
        n_dev = max(1, int(torch.cuda.device_count() or 1))
    except Exception:
        n_dev = 1
    if not S['both_gpus'] or n_dev < 2:
        if S['both_gpus'] and n_dev < 2:
            print('  note: asked for both T4s but this session has one card — running everything on cuda:0')
        return [dict(dev=0, keys=list(keys), units=list(keys))]
    njobs = min(2, n_dev)
    jobs = {d: dict(dev=d, keys=[], units=[]) for d in range(njobs)}
    for k in sorted(keys, key=lambda x: -cost[x]):
        d = min(jobs, key=lambda c: sum(cost.get(u, 0) for u in jobs[c]['units']))
        jobs[d]['keys'].append(k)
        jobs[d]['units'].append(k)
    if S['split']:
        idle = [d for d in jobs if not jobs[d]['units']]
        while idle:
            d0 = max(jobs, key=lambda c: sum(cost.get(u, 0) for u in jobs[c]['units']))
            cand = [k for k in jobs[d0]['units'] if k in INFO and _can_split(k)]
            if not cand:
                break
            k = max(cand, key=lambda x: cost.get(x, 0))
            parts = _split_into(k, 1 + len(idle))
            if not parts:
                break
            jobs[d0]['units'] = [u for u in jobs[d0]['units'] if u != k] + parts[:1]
            for b, pr in zip(idle, parts[1:]):
                jobs[b]['units'].append(pr)
            print(f'  {k}: split across cuda:{d0} and cuda:{idle[0]} as '
                  + ' + '.join(f'{p} ({INFO[p]["frames"]} f)' for p in parts)
                  + ' — the cut sits on a chunk boundary, and both halves got the same seed')
            idle = [d for d in jobs if not jobs[d]['units']]
    tot = {d: sum(cost.get(u, 0) for u in jobs[d]['units']) for d in jobs}
    print('cards: ' + ' · '.join(f'cuda:{d}: {len(jobs[d]["units"])} job item(s), '
                                 f'{tot[d]/1e6:.0f} Mpx-frames' for d in jobs if jobs[d]['units'])
          + f' — each process holds its own {_w_gb:.2f} GB of weights')
    return [j for j in jobs.values() if j['units']]


JOBS = _mk_jobs(items)
# 'keys' is what the report, STATE and resume track (whole files, parents of any split); 'units' is what
# the driver on that card actually runs — the same list, unless a clip was cut for the other card.
_fp_used = FP                                 # recomputed at the end if a retry changed the pixels
for ji, j in enumerate(JOBS):
    j['dir_in'] = os.path.join(QUEUE_IN, f'job{ji}')
    j['dir_out'] = os.path.join(DRV_OUT, f'job{ji}')
    j['tries'] = 0
    j['oom'] = False
    j['meas'] = []                              # per-frame cost measured on the card, pass by pass
    j['err'] = []                               # what the child said the file died of
    j['warn'] = []                              # raw OOM lines, for when it died before saying anything
    # 'keys' is the job for good (report, finalize, resume state); 'todo' is what still has to run.
    # A retry must never shrink the first list — that silently dropped every file the earlier pass
    # had already produced.
    j['todo'] = list(j['units'])
    for d in (j['dir_in'], j['dir_out']):
        shutil.rmtree(d, ignore_errors=True)
        os.makedirs(d, exist_ok=True)
    for k in j['todo']:
        link = os.path.join(j['dir_in'], k + os.path.splitext(INFO[k]['path'])[1].lower())
        try:
            os.symlink(INFO[k]['path'], link)
        except OSError:
            shutil.copy2(INFO[k]['path'], link)

def _driver_cmd(j):
    """The exact argv for one driver process. Flags mirror the fork's own CLI names where they exist, so a
    line printed here can be pasted into a shell against cli_main.py and mean the same thing."""
    # a half about to be redone must not later be joined with the leftovers of a previous attempt
    for u in list(j['todo']):
        for _f in glob.glob(os.path.join(j['dir_out'], u + '.*')):
            try:
                os.remove(_f)
            except OSError:
                pass
    a = [sys.executable, DRIVER, '--queue', j['dir_in'], '--out', j['dir_out'], '--repo', REPO_DIR,
         '--models_dir', MODELS_DIR, '--model', S['model'], '--mode', S['mode'], '--vae_model', S['vae'],
         '--precision', S['precision'], '--attention_mode', S['attention'], '--device', 'cuda:0',
         '--scale', str(S['scale']), '--resize_factor', f"{S['resize']:.3f}",
         '--attn_fallback', S['attn_fallback'],
         '--frame_chunk_size', str(max(1, S['chunk'])), '--auto_chunk', '1',
         '--probe_chunk', str(min(5, max(1, S['chunk']))), '--vram_headroom_mb', str(S['headroom']),
         '--tile_size', str(S['tile']),
         '--tile_overlap', str(S['tile_overlap']), '--sparse_ratio', f"{S['sparse_ratio']:.2f}",
         '--kv_ratio', f"{S['kv_ratio']:.2f}", '--local_range', str(S['local_range']),
         '--seed', str(S['seed']), '--color_fix', '1' if S['color_fix'] else '0',
         '--keep_models_on_cpu', '1' if S['offload'] else '0',
         '--start_sec', f"{S['start_sec']:.3f}", '--end_sec', f"{S['end_sec']:.3f}",
         '--max_frames', str(S['max_frames']), '--fps', f"{S['fps']:.6f}", '--vcodec', 'libx264',
         '--crf', str(S['crf']), '--preset', S['preset'], '--image_format', S['image_format'],
         '--image_quality', str(S['image_quality'])]
    if S['tiled_dit']:
        a.append('--tiled_dit')
    if S['tiled_vae']:
        a.append('--tiled_vae')
    if S['unload_dit']:
        a.append('--unload_dit')
    if S['debug']:
        a.append('--debug')
    if any(u in PARTWIN for u in j['todo']):
        # the halves of a split clip are cut in source frame numbers, so the join needs no re-timing
        wins = {u: PARTWIN[u] for u in j['todo'] if u in PARTWIN}
        wp = os.path.join(j['dir_in'], '_win.json')
        with open(wp, 'w', encoding='utf-8') as f:
            json.dump(wins, f)
        a += ['--windows', wp]
    if j.get('done_keys'):
        # tell the driver what a previous pass of THIS job already made, so a retry after an OOM does
        # not spend five minutes redoing the files that are sitting in its own output folder
        skip = os.path.join(j['dir_in'], '_skip.json')
        with open(skip, 'w', encoding='utf-8') as f:
            json.dump({k: 'already produced in an earlier pass' for k in j['done_keys']}, f)
        a += ['--skip', skip]
    return a


LOGS = [os.path.join(WORK, 'driver.log')] + [os.path.join(WORK, f'driver.cuda{j["dev"]}.log')
                                             for j in JOBS if j['dev'] != 0]
print('\nexact commands (one process per card, weights loaded once per process):')
for j in JOBS:
    print('  [cuda:%d] %s' % (j['dev'], ' '.join(_driver_cmd(j)[1:])))
    print(f'      {len(j["todo"])} file(s): ' + ', '.join(j['todo'][:6])
          + (' …' if len(j['todo']) > 6 else ''))
print('  logs: ' + ', '.join(os.path.basename(p) for p in LOGS))
if S['dry_run']:
    _say(f'Dry run: nothing was started (the plan and the commands are in {RUN_LOG}). '
         f'Tick "Dry run" off in a scratch cell — dry_w.value = False — to actually run.')
    RESULTS = [dict(_row(k, None), planned=f'chunk {S["chunk"]}, {S["scale"]}\u00d7{_sz}, '
                    f'{S["cost_px"]} B/px-frame, {INFO[k]["out"][0]}x{INFO[k]["out"][1]}')
               for j in JOBS for k in j['keys']]
    sys.exit(0)

# ------------------------------------------------------------------ 6. run and watch
RE_FILE = re.compile(r'Processing file\s+(\d+)/(\d+)')
RE_BATCH = re.compile(r'Upscaling batch\s+(\d+)/(\d+)')
RE_DONE = re.compile(r'\[?([\w.\-]+)\]?\s*completed successfully in ([\d.]+)s')
RE_SAVE = re.compile(r'Output saved to:\s*(\S+)')
RE_VRAM = re.compile(r'Peak VRAM Used:\s*([\d.]+)\s*GB')
# The driver is the only thing here that ever sees the real card, so its readings belong in the report:
# the per-frame cost it measures (that is what the chunk is fitted from), the stats it ends on, and the
# error it died of — without the last one, a non-OOM failure in a child shows up as nothing but "exit 1".
RE_MEAS = re.compile(r'MEASURED probe=(\d+) frames=(\d+) per_frame_mb=([\d.]+) free_mb=([\d.]+) '
                     r'headroom_mb=([\d.]+) cap_frames=(\d+)')
RE_STATS = re.compile(r'^STATS (\{.*\})$')
RE_ERR = re.compile(r'^ERROR in ([\w.\-]+): (.*)$')
HIST = {}


def _job_frames(j):
    """(total, finished) frames for one card. Frames rather than files: one 900-frame clip on its own card
    would otherwise spend the whole run showing 0 % and then 100 %, which is not a progress bar."""
    u = j.get('units') or j.get('todo') or []
    tot = float(sum(max(1, int(INFO[x]['frames'])) for x in u if x in INFO))
    done = float(sum(max(1, int(INFO[x]['frames'])) for x in (j.get('done_keys') or []) if x in INFO))
    cur = j.get('cur')
    if cur in INFO and cur not in (j.get('done_keys') or []):
        frac = min(1.0, max(0.0, j.get('batch_i', 0) / max(1, j.get('batch_n', 1))))
        done += int(INFO[cur]['frames']) * frac
    return max(1.0, tot), min(tot, done)


def _status():
    """One line for the run as a whole: frames done, time gone, and what the cards say is left."""
    try:
        tot = done = 0.0
        lefts, rates = [], []
        for jj in JOBS:
            t, d = _job_frames(jj)
            tot += t
            done += d
            if jj.get('rate'):
                rates.append(jj['rate'])
                lefts.append((t - d) * jj['rate'])
        el = time.time() - float(globals().get('_t_all') or time.time())
        est = max(lefts) if lefts else None
        STATUS.value = ('<div style="font-family:monospace;font-size:12px;color:#334">all cards: <b>'
                        + f'{100*done/max(1.0,tot):.0f}%</b> of {tot:.0f} frames · {fmt_dur(el)} gone · '
                        + ('about ' + fmt_dur(est) + ' left' if est is not None else 'measuring speed…')
                        + (f' · {min(rates):.2f}-{max(rates):.2f} s/frame' if rates else '')
                        + f' · {len(JOBS)} card(s) running</div>')
    except Exception:
        pass                                   # a half-built UI during a retry is not worth a traceback


def _tick(j):
    tot, done = _job_frames(j)
    p = max(0.0, min(1.0, done / tot))
    j['bar'].value = p
    el = time.time() - float(j.get('t_start') or time.time())
    rate = j.get('rate')
    left = (tot - done) * rate if rate else None
    n = max(1, len(j.get('units') or []))
    fi = int(j.get('file_i') or 0)
    txt = (f'cuda:{j["dev"]} · {p*100:.0f}% · {fmt_dur(el)} gone · '
           + (f'~{fmt_dur(left)} left' if left is not None else 'estimating\u2026')
           + f' · {done:.0f}/{tot:.0f} fr · file {fi}/{n} ' + str(j.get('cur') or '')[:24]
           + f' · pass {j.get("batch_i", 0)}/{j.get("batch_n", 1)}'
           + (f' · {rate:.2f} s/fr' if rate else '')
           + (f' · peak {j["peak"]:.1f} GB' if j.get('peak') else ''))
    j['html'].value = f'<span style="font-family:monospace;font-size:12px">{txt}</span>'
    _status()


def _on_line(line, j, logf):
    logf.write(line + '\n')
    try:
        logf.flush()
    except Exception:
        pass
    m = RE_FILE.search(line)
    if m:
        j['file_i'], j['batch_i'], j['batch_n'] = int(m.group(1)), 0, 1
        _i = int(m.group(1)) - 1
        j['cur'] = j['todo'][_i] if _i < len(j['todo']) else '?'
        j['t0'] = time.time()
        _tick(j)
        return
    m = RE_BATCH.search(line)
    if m:
        j['batch_i'], j['batch_n'] = int(m.group(1)), int(m.group(2))
        _tick(j)
        return
    m = RE_VRAM.search(line)
    if m:
        j['peak'] = float(m.group(1))
        return
    m = RE_SAVE.search(line)
    if m:
        j.setdefault('saved', []).append(m.group(1))
        return
    m = RE_DONE.search(line)
    if m and m.group(1) in INFO:
        key, secs = m.group(1), float(m.group(2))
        fr = max(1, INFO[key]['frames'])
        j['rate'] = secs / fr
        HIST[key] = dict(seconds=round(secs, 2), frames=fr, peak_gb=round(j.get('peak') or 0.0, 2),
                         saved=j['saved'][-1] if j.get('saved') else None)
        j.setdefault('done_keys', []).append(key)
        _tick(j)
        return
    m = RE_MEAS.search(line)
    if m:
        j['meas'].append(dict(probe=int(m.group(1)), frames=int(m.group(2)), per_frame_mb=float(m.group(3)),
                              free_mb=float(m.group(4)), headroom_mb=float(m.group(5)),
                              cap_frames=int(m.group(6))))
        return
    m = RE_STATS.search(line)
    if m:
        try:
            j['stats'] = json.loads(m.group(1))
        except ValueError:
            pass
        return
    m = RE_ERR.match(line.strip())
    if m:
        if len(j['err']) < 12:
            j['err'].append(f'{m.group(1)}: {m.group(2)}'[:220])
    if 'out of memory' in line.lower() or 'OutOfMemoryError' in line:
        j['oom'] = True
        if len(j['warn']) < 4:
            j['warn'].append(line.strip()[:180])


def _pump(proc, logf, j):
    try:
        for line in iter(proc.stdout.readline, ''):
            _on_line(line.rstrip('\n'), j, logf)
    except Exception as exc:                         # the log pipe can die with the child on Ctrl+C
        print(f'  ! reader stopped: {exc}')


def _kill(pr):
    if pr.poll() is None:
        for sig, wait in ((15, 4), (9, 1)):
            try:
                os.killpg(os.getpgid(pr.pid), sig)
            except Exception:
                return
            time.sleep(wait)
            if pr.poll() is not None:
                return


def _spawn(j, logmode):
    lf = open(j['logp'], logmode, encoding='utf-8')
    env = dict(os.environ)
    env['CUDA_VISIBLE_DEVICES'] = str(j['dev'])
    env['PYTHONUNBUFFERED'] = '1'
    env['HF_HUB_DISABLE_TELEMETRY'] = '1'
    env['PYTORCH_CUDA_ALLOC_CONF'] = os.environ.get('PYTORCH_CUDA_ALLOC_CONF',
                                                     'expandable_segments:True')
    j.setdefault('t_start', time.time())
    pr = subprocess.Popen(_driver_cmd(j), cwd=WORK, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    j['proc'], j['logf'] = pr, lf
    threading.Thread(target=_pump, args=(pr, lf, j), daemon=True).start()
    return pr


STATUS = W.HTML('')
display(STATUS)
for j, lp in zip(JOBS, LOGS):
    j['logp'] = lp
    j['done_keys'] = []
    j['t_start'] = time.time()
    j['bar'] = W.FloatProgress(value=0.0, min=0.0, max=1.0, bar_style='info',
                              layout=W.Layout(width='600px'))
    j['html'] = W.HTML('')
    display(W.HBox([j['bar'], j['html']]))

_t_all = time.time()
try:
    for j in JOBS:
        _spawn(j, 'w')
    for j in JOBS:
        j['rc'] = j['proc'].wait()
        j['logf'].close()
except KeyboardInterrupt:
    for j in JOBS:
        _kill(j['proc'])
    print('\n^C — the driver was killed. Files that had already printed their result were finalized.')
finally:
    for j in JOBS:
        try:
            j['logf'].close()
        except Exception:
            pass

# ------------------------------------------------------------------ 7. retry what OOMed
def _refit(j):
    """The driver already shrank this file's chunk by measurement and the card still said no, so what is
    left is an activation peak inside the pass, the pixels themselves, or the ceiling it was handed.
    Every step below keeps the output geometry identical until the last two, which are said out loud."""
    if not (S['tiled_dit'] and S['tiled_vae']):
        S['tiled_dit'] = S['tiled_vae'] = True
        S['cost_px'] = fv_bytes_per_outpx(S['mode'], True, True)
        _fix(f'both tilers forced on (cuda:{j["dev"]})',
             'the OOM happened inside a pass the driver could not shrink further; tiling is what cuts the '
             'peak of a single frame, the chunk is not')
        return True
    if not (S['offload'] and S['unload_dit']):
        _o, _u = S['offload'], S['unload_dit']
        S['offload'], S['unload_dit'] = True, True
        _fix(f'"Keep weights on CPU" on, "Unload DiT after each file" on (cuda:{j["dev"]})',
             f'the card is short by about the {_GB:.1f} GB of weights it keeps resident; offloading them '
             'costs one read from disk per file and nothing else')
        return True
    if S['kv_ratio'] > 1.0 or S['local_range'] > 9 or S['sparse_ratio'] < 1.5:
        _k, _lr = S['kv_ratio'], S['local_range']
        S['kv_ratio'], S['local_range'], S['sparse_ratio'] = 1.0, min(9, S['local_range']), \
            max(1.5, S['sparse_ratio'])
        _fix(f'attention windows kv_ratio {_k:g} → 1, local_range {_lr} → {S["local_range"]}',
             'the streaming KV cache is the other term that grows with the chunk; this trims it. It can '
             'soften the last frames of a long clip — that is the trade, not a bug')
        return True
    if S['resize'] > 0.35:
        _r = S['resize']
        S['resize'] = round(max(0.35, S['resize'] * 0.75), 3)
        _fix(f'pre-shrink input {_r} → {S["resize"]} (cuda:{j["dev"]})',
             'the card is short by more than tilers can give, so the output pixels themselves come down — at '
             '0.75 they are 43% smaller, and every remaining file is produced at that same size')
        return True
    if S['chunk'] > 1:
        _c = S['chunk']
        S['chunk'] = max(1, S['chunk'] // 2)
        S['cost_px'] = fv_bytes_per_outpx(S['mode'], S['tiled_dit'], S['tiled_vae'])
        _fix(f'chunk ceiling {_c} → {S["chunk"]}',
             'the driver had already fitted the chunk to this card by measurement and still hit an OOM, so '
             'the ceiling it works under is halved outright')
        return True
    return False


_MEMISH = ('out of memory', 'exceeds this card', 'failed to alloc', 'not enough memory',
           'cublas_status_alloc')


def _memory_failure(j):
    """Only the card saying "no room" earns a smaller-settings retry.

    The driver now fits its own chunk, so a restart with halved numbers is a last resort for the passes
    it could not fix itself. Everything else — a kernel this GPU cannot build, an unreadable file, an
    assert in the pipeline — is not a size problem, and shrinking pixels to chase it would quietly
    produce wrong-sized output that the next run then refuses to redo.
    """
    if j.get('oom'):
        return True
    txt = ' '.join((j.get('err') or []) + (j.get('warn') or [])).lower()
    return any(k in txt for k in _MEMISH)


for j in JOBS:
    while _memory_failure(j) and len(j.get('done_keys') or []) < len(j['todo']) and j['tries'] < 2:
        j['tries'] += 1
        left = [k for k in j['todo'] if k not in (j.get('done_keys') or [])]
        if not left:
            break
        if not _refit(j):
            print(f'  ! cuda:{j["dev"]} is at chunk 1 with both tilers and a shrunk input and still OOMs — '
                  f'leaving {len(left)} file(s) undone (' + ', '.join(left[:4]) + ')')
            break
        j['oom'] = False
        _recalc_dims(f'sizes recomputed after the retry (pre-shrink {S["resize"]})'
                     if S['resize'] != _RESIZE_ASKED else '')
        j['todo'] = left
        j['saved'] = []
        shutil.rmtree(j['dir_in'], ignore_errors=True)
        os.makedirs(j['dir_in'], exist_ok=True)
        for k in left:
            try:
                os.symlink(INFO[k]['path'], os.path.join(j['dir_in'], k + os.path.splitext(INFO[k]['path'])[1]))
            except OSError:
                pass
        print(f'\nretrying cuda:{j["dev"]} with chunk {S["chunk"]}, tiling {S["tiled_dit"]}/'
              f'{S["tiled_vae"]}, pre-shrink {S["resize"]} on {len(left)} file(s)')
        _spawn(j, 'a')
        j['rc'] = j['proc'].wait()
        j['logf'].close()

# ------------------------------------------------------------------ 8. finalize: audio, name, done-marker
OUT_V_EXT = {'copy': '.mp4', 'mp4': '.mp4', 'mkv': '.mkv', 'webm': '.webm'}


def _free_name(stem, ext):
    p = os.path.join(OUTPUT_DIR, stem + ext)
    i = 1
    while os.path.exists(p):
        p = os.path.join(OUTPUT_DIR, f'{stem}_{i}{ext}')
        i += 1
    return p


def _has_audio(path):
    if not (FFPROBE and path and os.path.isfile(path)):
        return False
    try:
        r = subprocess.run([FFPROBE, '-v', 'error', '-select_streams', 'a', '-show_entries',
                            'stream=index', '-of', 'csv=p=0', path], capture_output=True, text=True)
        return bool(r.stdout.strip())
    except Exception:
        return False


mux_notes = []
join_notes = []                # one line per clip that was cut across the cards, so the join is not silent
for j in JOBS:
    for k in list(j['todo']):
        if k in (j.get('done_keys') or []):
            continue
        for f in sorted(glob.glob(os.path.join(j['dir_out'], k + '.*'))):
            if os.path.getsize(f) > 0:                      # a killed run can still leave a real file
                HIST.setdefault(k, dict(seconds=None, frames=INFO[k]['frames'], peak_gb=None, saved=f))
                j.setdefault('done_keys', []).append(k)
                break

def _fit_for(k, j):
    """What the driver says this file cost — summed over its halves when it ran as a split clip."""
    f = (j.get('stats') or {}).get(k)
    if f or k not in SPLIT:
        return f or {}
    acc = dict(passes=0, ooms=0, chunk=10 ** 9, per_frame_mb=0.0, seconds=0.0)
    for pk in SPLIT[k]:
        for jj in JOBS:
            st = (jj.get('stats') or {}).get(pk) or {}
            if not st:
                continue
            acc['passes'] += st.get('passes', 0)
            acc['ooms'] += st.get('ooms', 0)
            acc['chunk'] = min(acc['chunk'], int(st.get('chunk') or acc['chunk']))
            acc['per_frame_mb'] = max(acc['per_frame_mb'], st.get('per_frame_mb') or 0.0)
            acc['seconds'] += st.get('seconds') or 0.0
            acc['attn'] = st.get('attn') or acc.get('attn')
    if acc['chunk'] == 10 ** 9:
        acc.pop('chunk')
    acc['seconds'] = round(acc['seconds'], 2)
    return acc if acc.get('passes') else {}


def _assemble_parts(k, j):
    """Join the halves of a split clip into the one file the rest of the cell expects.

    -c copy first: both halves came out of the same encoder at the same size and fps, so the concat
    demuxer splices them with no second lossy pass. If the frame count that comes back is not the sum of
    the parts, the GOP boundaries disagreed and the join is re-encoded instead — a re-encode at the
    configured CRF still beats a file that lost a frame in the middle."""
    parts = SPLIT.get(k) or []
    found = {}
    for jj in JOBS:
        for pk in parts:
            f = os.path.join(jj['dir_out'], pk + '.mp4')
            if pk not in found and os.path.isfile(f) and os.path.getsize(f) > 0:
                found[pk] = f
    if len(found) != len(parts):
        return None, 'no output for ' + ', '.join(pk for pk in parts if pk not in found)
    ordered = [found[pk] for pk in sorted(found, key=lambda x: INFO[x]['part'])]
    exp = sum(int(INFO[pk]['frames']) for pk in parts)
    dst = os.path.join(j['dir_out'], k + '.mp4')
    lst = dst + '.list.txt'
    with open(lst, 'w', encoding='utf-8') as f:
        for pth in ordered:
            f.write("file '" + pth.replace("'", "'\\''") + "'\n")
    r = subprocess.run([FFMPEG, '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', lst, '-c', 'copy',
                        dst], capture_output=True, text=True)
    how, got = 'stream-copied', int((probe_video(dst) or {}).get('n_frames') or 0)
    if r.returncode or not os.path.isfile(dst) or (got and abs(got - exp) > 1):
        r = subprocess.run([FFMPEG, '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', lst, '-c:v',
                            'libx264', '-crf', str(S['crf']), '-preset', S['preset'], '-pix_fmt',
                            'yuv420p', dst], capture_output=True, text=True)
        how, got = 're-encoded', int((probe_video(dst) or {}).get('n_frames') or 0)
    try:
        os.remove(lst)
    except OSError:
        pass
    if r.returncode or not os.path.isfile(dst) or os.path.getsize(dst) == 0:
        return None, (r.stderr or '').strip()[-160:] or 'ffmpeg concat failed'
    if got:
        INFO[k]['frames'] = got
    note = f'joined {len(parts)} halves across cards ({how}, {got or exp} frames)'
    if got and abs(got - exp) > 1:
        note += f' \u2014 note: {exp} frames were produced, {got} survived the join'
    return dst, note


def _finalize_video(src_file, dst_file, key):
    """Container + audio in ONE ffmpeg pass.

    The driver already encoded H.264 at the CRF from Config, so "keep the driver's MP4" and "MP4" are
    just an audio mux; MKV is a lossless remux of that; WebM is the only case that re-encodes, and it
    pulls the audio from the source in the same command (a VP9 pass with -an used to drop the track)."""
    info = INFO[key]
    want_a = bool(S['audio']) and _has_audio(info['path'])
    trim_pre = ['-ss', f"{S['start_sec']:.3f}"] if S['start_sec'] > 0 else []
    trim_dur = ['-t', f'{S["end_sec"] - S["start_sec"]:.3f}'] if S['end_sec'] > S['start_sec'] else []
    if S['vfmt'] in ('copy', 'mp4'):
        if not want_a:
            shutil.copy2(src_file, dst_file)
            return 'copied — the driver wrote x264 at CRF %d already' % S['crf']
        cmd = ([FFMPEG, '-y', '-v', 'error', '-i', src_file] + trim_pre + ['-i', info['path'],
               '-map', '0:v:0', '-map', '1:a:0', '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k']
               + trim_dur + ['-shortest', dst_file])
    elif S['vfmt'] == 'mkv':
        cmd = [FFMPEG, '-y', '-v', 'error', '-i', src_file] + trim_pre
        if want_a:
            cmd += ['-i', info['path'], '-map', '0:v:0', '-map', '1:a:0', '-c:v', 'copy', '-c:a', 'aac']
        else:
            cmd += ['-map', '0:v:0', '-c', 'copy']
        cmd += trim_dur + ['-shortest', dst_file]
    else:                                              # webm: VP9 re-encode, audio kept
        cmd = [FFMPEG, '-y', '-v', 'error', '-i', src_file] + trim_pre
        if want_a:
            cmd += ['-i', info['path'], '-map', '0:v:0', '-map', '1:a:0', '-c:a', 'libopus', '-b:a', '128k']
        else:
            cmd += ['-map', '0:v:0', '-an']
        cmd += ['-c:v', 'libvpx-vp9', '-crf', str(min(50, S['crf'] + 10)), '-b:v', '0',
                '-pix_fmt', 'yuv420p'] + trim_dur + [dst_file]
    if not cmd:
        shutil.copy2(src_file, dst_file)
        return 'copied'
    if os.path.exists(dst_file):
        os.remove(dst_file)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0 and os.path.isfile(dst_file) and os.path.getsize(dst_file) > 0:
        return ('muxed audio' if want_a and S['vfmt'] in ('copy', 'mp4', 'mkv') else
                'vp9 + audio' if want_a else ('remuxed to mkv' if S['vfmt'] == 'mkv' else 're-encoded'))
    if os.path.isfile(dst_file) and os.path.getsize(dst_file) == 0:
        os.remove(dst_file)
    shutil.copy2(src_file, dst_file)
    return 'ffmpeg fell back to the driver file: ' + (r.stderr.strip()[-160:] or 'non-zero exit')


# What the driver ended up running, once it had looked at the card: the attention kernel and the pixel
# size can both differ from what was planned, and both change the picture, so both belong in the
# fingerprint these files are remembered under.
attn_used = sorted({(st or {}).get('attn', '') for j in JOBS for st in (j.get('stats') or {}).values()})
attn_used = [a for a in attn_used if a]
_dense_run = next((a for a in attn_used if 'dense' in a), None)
_fp_used = fingerprint(**{k: v for k, v in S.items() if k not in FP_IGNORE})
RESULTS, failed = [], []
for j in JOBS:
    produced = {}
    for f in sorted(glob.glob(os.path.join(j['dir_out'], '*'))):
        if os.path.getsize(f) > 0:
            produced.setdefault(os.path.splitext(os.path.basename(f))[0], f)
    for k in j['keys']:
        jn = ''
        src = produced.get(k)
        if k in SPLIT:
            src, jn = _assemble_parts(k, j)     # the halves live on different cards' output dirs
            if jn:
                join_notes.append(f'{k}: {jn}')
        if not src:
            _said = next(iter(j.get('err') or []), '')
            why = ('CUDA out of memory even after in-process shrinking' if j.get('oom') else
                   (_said[:160] if _said else f'driver exit {j.get("rc")}') if j.get('rc')
                   else 'no output file (interrupted?)')
            if k in SPLIT:
                why = 'a half of the split clip never came out (' + (jn or 'no output') + ')'
            failed.append((k, why))
            continue
        kind = INFO[k]['kind']
        _fit = _fit_for(k, j)
        # the extension follows the file the driver ACTUALLY wrote (image_format "same" keeps .jpg,
        # and a picture must not be renamed to .png with JPEG bytes inside); only the video containers
        # that need a second pass get the mapped one.
        real_ext = os.path.splitext(src)[1].lower() or '.mp4'
        ext = (real_ext if kind == 'image' or S['vfmt'] in ('copy', 'mp4')
               else OUT_V_EXT.get(S['vfmt'], '.mp4'))
        out = _free_name(k, ext)
        note = ''
        if kind == 'video':
            note = _finalize_video(src, out, k)
            if jn:
                note = jn + '; ' + note
            if note.startswith('ffmpeg fell back'):
                mux_notes.append(f'{k}: {note}')
        else:
            shutil.copy2(src, out)
        dw, dh = INFO[k]['out']
        # fp = the fingerprint these files' PIXELS answer to. A run that had to pre-shrink the input made
        # something smaller than the settings asked for, so it is recorded under its own fingerprint and a
        # later run with the original settings redoes it, instead of skipping a wrong-size file forever.
        if _dense_run:
            note = (note + '; ' if note else '') + 'ran on dense SDPA — this card cannot build the ' \
                                                  'sparse kernel this fork insists on'
        STATE[k] = dict(fp=_fp_used, fp_planned=FP, out=os.path.basename(out), kind=kind, size_out=[dh, dw],
                        seconds=HIST.get(k, {}).get('seconds') or _fit.get('seconds'),
                        when=time.strftime('%Y-%m-%d %H:%M'),
                        note=note,
                        fit={kk: _fit.get(kk) for kk in ('passes', 'chunk', 'ooms', 'per_frame_mb', 'attn')
                             if _fit.get(kk) is not None})
        RESULTS.append(_row(k, out))
        if S['cleanup']:
            try:
                os.remove(src)
            except OSError:
                pass
with open(STATE_PATH, 'w', encoding='utf-8') as f:
    json.dump(STATE, f, indent=1, sort_keys=True)
for j in JOBS:
    shutil.rmtree(j['dir_in'], ignore_errors=True)
    shutil.rmtree(j['dir_out'], ignore_errors=True)

# ------------------------------------------------------------------ 9. report
_secs = time.time() - _t_all
print(f'\n{"="*80}\nFlashVSR: {len(RESULTS)} file(s) done in {fmt_dur(_secs)} — '
      f'{len(JOBS)} driver process(es), chunk {S["chunk"]}, {S["scale"]}\u00d7{_sz}, '
      f'{S["mode"]}, {"fp16" if S["precision"] != "bf16" else "bf16"}')
for r in sorted(RESULTS, key=lambda x: x['out']):
    print(f'  {r["out_name"]:<40} {r["w"]}x{r["h"]:<7} {r["frames"]:>5} f '
          f'{(str(round(r["seconds"], 2)) + "s") if r["seconds"] else "?":>9} {fmt_size(r["size"]):>10}'
          + (f'  peak {r["peak_gb"]:.1f} GB' if r.get('peak_gb') else ''))
for k, why in failed:
    print(f'  FAILED {k}: {why}')
if SPLIT:
    print('  split across cards: ' + ', '.join(
        f'{k} = ' + ' + '.join(str(INFO[p]['frames']) + 'f' for p in SPLIT[k]) for k in SPLIT)
        + ' (cut on a chunk boundary, same seed on both halves, joined without a re-encode)')
for note in join_notes:
    print('  ' + note)
for note in mux_notes:
    print('  ' + note)
meas = [m for j in JOBS for m in j.get('meas') or []]
if meas:
    print('\nmeasured on the card — this is what the chunk was fitted from, not the formula:')
    for m in meas[:6]:
        print(f'  · a pass of {m["frames"]} frame(s) peaked at '
              f'{m["per_frame_mb"] * m["frames"] / 1024:.2f} GB = {m["per_frame_mb"]:.0f} MB per frame · '
              f'{m["free_mb"] / 1024:.2f} GB free on the card with {m["headroom_mb"] / 1024:.2f} GB held '
              f'back → up to {m["cap_frames"]} frame(s) per pass')
    if len(meas) > 6:
        print(f'  · … and {len(meas) - 6} more measurement pass(es) in the logs')
elif any(j.get('err') or j.get('warn') for j in JOBS):
    print('\nnote: nothing was measured because no pass ever finished — the driver reports its error '
          'below, and that is what to read, not the chunk size')
else:
    print('\nnote: the driver could not measure on this card (torch.cuda readings unavailable), so the '
          'chunk stayed where the formula put it')
errs = list(dict.fromkeys(e for j in JOBS for e in j.get('err') or []))
warns = list(dict.fromkeys(w for j in JOBS for w in j.get('warn') or []))
if any('dense' in a for a in attn_used):
    print('\nattention: dense SDPA — this build\u2019s own sparse kernel does not run on this card, so the '
          'driver dropped the block mask and every tile got full attention. Slower per frame, and the only '
          'way a T4 has been seen to finish; the sparse kernel needs Ampere and triton<3.3.0.')
if errs or warns:
    print('\nthe driver reported:')
    for e in (errs + warns)[:4]:
        print('  ! ' + e)
    if len(errs) + len(warns) > 4:
        print(f'  ! \u2026 and {len(errs) + len(warns) - 4} more line(s) in '
              + ', '.join(os.path.basename(q) for q in LOGS))
    for sig, hint in (('passmanager',
                       'this is the fork\u2019s own block-sparse Triton kernel, not a memory problem: the '
                       'build routes every self-attention call into it whatever "Attention" says. The '
                       'driver switches to dense SDPA by itself, so if this line survived, '
                       '"Enforce dense SDPA" is unticked (tick it) — or keep the sparse kernel and install '
                       'triton<3.3.0, the recipe the fork gives for Turing cards'),
                      ('no kernel image',
                       'the torch build in this session does not match this GPU — rerun the Install cell, '
                       'which pins the torch that does'),
                      ('invalid data found',
                       'a frame the decoder cannot read: trim the clip or re-encode it, the chunk size has '
                       'nothing to do with it')):
        if any(sig in x.lower() for x in (errs + warns)):
            print('  \u2192 ' + hint)
if FIX:
    print('\nwhat this cell changed, and why:')
    for what, why in FIX:
        print(f'  · {what} → {why}')
if any(INFO[k]['kind'] == 'video' for k in items) and S['mode'] != 'full':
    print('\nsoft output at the end of a long clip? that is the chunk boundary, not a bug: the pipeline pads '
          'each chunk to 8n+5 frames by repeating its last frame. Raise "Frames per chunk" or lower '
          '"kv_ratio" for less of it, at the price of memory.')
print(f'\nresults in {OUTPUT_DIR} · logs in {", ".join(os.path.basename(q) for q in LOGS)} · '
      f'fingerprint {_fp_used} (the next run with these exact settings resumes from it'
      + (f'; it was planned as {FP} — this run had to use different pixels, so what it made will be '
         'redone when the settings as asked are affordable' if _fp_used != FP and RESULTS else '')
      + (f'; note that these files were made with {_dense_run}' if _dense_run else '') + ')')
print('HIST: ' + json.dumps(HIST, sort_keys=True))

# the one line this cell leaves on the screen: what came out, what did not, and where the whole text is
_say(f'{len([r for r in RESULTS if r.get("out")])} file(s) done in {fmt_dur(_secs)}, {len(failed)} failed'
     + (f' · {len(FIX)} setting(s) adjusted' if FIX else '')
     + f' · results: {OUTPUT_DIR} · everything this run said: {RUN_LOG}')
print = _real_print                                  # the shadow belongs to this cell only
if _run_log is not None:
    _run_log.close()
    globals()['_run_log'] = None


In [ ]:
#@title ##**Visualization** { display-mode: "form" }
# title: Visualization
# The pictures, in the style of the Real-ESRGAN notebook: one dropdown, two ways to look at a result.
#
#   Side-by-Side — the source and the result are resized to the same height and pasted into ONE image, left
#                  and right, and shown as an ordinary notebook image (cheap, scrollable, works with any
#                  viewer, and you can select it and copy it).
#   Slider       — both are laid exactly on top of each other with a white handle you drag: left of the
#                  handle is the source, right of it is the result. This is the one that shows what actually
#                  changed: the same edge under both halves, no guessing about scale.
#
# A video is compared on a single frame taken from the middle of each file (the middle is where a clip has
# both motion and detail; change PREVIEW_POS in a scratch cell for another moment), and — when the result is
# small enough to embed — also shown as two players next to each other, so you can watch the same moment
# move. MP4/H.264 and WebM play inline in the browser; MKV usually does not, so for those use the comparison
# and the Download Results cell.
#
# Nothing is written next to your results: frames are extracted into /content/vis_tmp, which is deleted when
# the cell finishes. Both pictures are embedded as base64, which is why there are two caps below — a 4000×3000
# PNG pasted six times is what makes a Colab tab swap to disk. EMBED_LIMIT_MB only guards the video players.
import os, io, base64, shutil, subprocess

import PIL.Image
from IPython.display import HTML, display

if 'RESULTS' not in globals() or not RESULTS:
    raise RuntimeError('No results found — run the Run cell first. '
                       '(The Download Results cell works without this one.)')
if 'FFMPEG' not in globals() or not FFMPEG:
    raise RuntimeError('Run the Install cell first — it locates the ffmpeg used to pull a frame out of a clip.')

visualization_method = "Side-by-Side"  #@param ["Side-by-Side", "Slider"]
#   Side-by-Side = one composed image, source left / result right. Slider = a draggable handle over the pair.

PREVIEW_WIDTH = 900        # comparison stills are scaled to at most this width before embedding
PREVIEW_POS = 0.5          # which moment of a clip the comparison frame is taken from (0..1)
EMBED_LIMIT_MB = 25        # a bigger video is not pasted into the notebook; the comparison still is shown
MAX_ITEMS = 6              # at most this many blocks per run of the cell

VIS_TMP = f'{WORK}/vis_tmp'
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tif', '.tiff')
VIDEO_MIME = {'.mp4': 'video/mp4', '.m4v': 'video/mp4', '.webm': 'video/webm', '.mov': 'video/quicktime',
              '.mkv': 'video/x-matroska'}


def is_image_file(filename):
    """The reference notebook's helper, kept name for name."""
    return os.path.splitext(filename.lower())[1] in IMG_EXTS


def resize_image_maintain_aspect(image, max_width, target_height=None):
    """Shrink to a width, keeping the aspect; then match a height if one is given. Reference code."""
    width, height = image.size
    if width > max_width:
        image = image.resize((max_width, int(height * max_width / width)), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        image = image.resize((int(image.size[0] * target_height / image.size[1]), target_height),
                             PIL.Image.Resampling.LANCZOS)
    return image


def image_to_base64(image):
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')


def _pair_list():
    """(source, result, label) for everything this batch produced, in the order the Run cell reported it.
    RESULTS is the honest source — the names of the outputs are the driver's business, not something to
    guess by sorting two folders. If the runtime restarted and only the files are left, fall back to
    matching a result against the upload it came from by name."""
    out = []
    for r in (globals().get('RESULTS') or []):
        src, dst = r.get('src') or '', r.get('out') or ''
        if dst and os.path.isfile(dst):
            out.append((src if os.path.isfile(src) else None, dst, os.path.basename(dst)))
    if out:
        return out[:MAX_ITEMS]
    res = sorted(os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR)
                 if os.path.splitext(f.lower())[1] in IMG_EXTS + tuple(VIDEO_MIME))
    srcs = sorted(os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR)) if os.path.isdir(INPUT_DIR) else []
    for q in res[:MAX_ITEMS]:
        stem = os.path.splitext(os.path.basename(q))[0]
        match = next((p for p in srcs if os.path.splitext(os.path.basename(p))[0] in stem), None)
        out.append((match, q, os.path.basename(q)))
    return out


def _grab_frame(video_path, at):
    """One frame of a clip, as a PIL image, taken at `at` of its length. ffmpeg writes it into VIS_TMP and
    the temp file is deleted right here: a comparison should not leave anything behind next to the results."""
    os.makedirs(VIS_TMP, exist_ok=True)
    secs = max(0.0, float(at))
    dst = os.path.join(VIS_TMP, f'f{abs(hash(video_path)) % 99999}.png')
    subprocess.run([FFMPEG, '-y', '-hide_banner', '-loglevel', 'error', '-ss', f'{secs:.3f}', '-i', video_path,
                    '-frames:v', '1', dst], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    try:
        return PIL.Image.open(dst).convert('RGB')
    finally:
        try:
            os.remove(dst)
        except OSError:
            pass


def _duration(path):
    try:
        info = probe_video(path)
        n, fps = float(info.get('n_frames') or 0), float(info.get('fps') or 0)
        return n / fps if (n and fps) else 0.0
    except Exception:
        return 0.0


def _load(path, at_mid=PREVIEW_POS):
    """The picture to compare: the file itself for a still, the middle frame for a clip."""
    if is_image_file(path):
        return PIL.Image.open(path).convert('RGB')
    dur = _duration(path)
    return _grab_frame(path, dur * at_mid if dur else 0.0)


def _side_by_side(before, after, label):
    """The reference cell's own recipe: match the heights, paste into one image, display it."""
    max_width = 500
    b = resize_image_maintain_aspect(before.copy(), max_width)
    a = resize_image_maintain_aspect(after.copy(), max_width)
    h = min(b.size[1], a.size[1])
    b = resize_image_maintain_aspect(b, max_width, h)
    a = resize_image_maintain_aspect(a, max_width, h)
    combined = PIL.Image.new('RGB', (b.size[0] + a.size[0] + 8, h), (24, 24, 24))
    combined.paste(b, (0, 0))
    combined.paste(a, (b.size[0] + 8, 0))
    display(combined)          # and nothing else: the reference cell prints no caption between results


def _slider(before, after, label):
    """Both on the same rectangle, the result clipped to the right of a draggable handle (the reference
    cell's mechanism: clip-path + a mousedown on the divider, no widget library involved)."""
    w = min(after.size[0], PREVIEW_WIDTH)
    a = resize_image_maintain_aspect(after.copy(), w)
    h = a.size[1]
    b = resize_image_maintain_aspect(before.copy(), w, h)
    uid = f'v{abs(hash(label + str(len(label)))) % 100000}'
    html = f"""
<div style="font-family:sans-serif;width:{w}px;margin:6px 0 26px 0">
  <div style="font-size:13px;margin-bottom:4px"><b>{label}</b>
       <span style="color:#777"> — drag the white line; left is the source, right is the result</span></div>
  <div id="{uid}" style="position:relative;width:100%;height:{h}px;overflow:hidden;cursor:ew-resize;
                         user-select:none;background:#111;border-radius:4px">
    <img src="data:image/png;base64,{image_to_base64(b)}" style="position:absolute;left:0;top:0;
         width:100%;height:100%">
    <div class="clip" style="position:absolute;left:0;top:0;width:100%;height:100%;overflow:hidden;
                             clip-path:inset(0 0 0 50%)">
      <img src="data:image/png;base64,{image_to_base64(a)}" style="position:absolute;left:0;top:0;
           width:100%;height:100%">
    </div>
    <div class="bar" style="position:absolute;top:0;bottom:0;left:50%;width:4px;background:#fff;
                           box-shadow:0 0 5px rgba(0,0,0,.6)">
      <div style="position:absolute;top:50%;left:-8px;width:20px;height:20px;transform:translateY(-50%);
                  background:#fff;border-radius:50%"></div>
    </div>
    <div style="position:absolute;top:8px;left:8px;color:#fff;font-size:13px;
                text-shadow:0 1px 3px rgba(0,0,0,.9)">Source</div>
    <div style="position:absolute;top:8px;right:8px;color:#fff;font-size:13px;
                text-shadow:0 1px 3px rgba(0,0,0,.9)">FlashVSR</div>
  </div>
</div>
<script>
(function(){{
  var root = document.getElementById('{uid}');
  var clip = root.querySelector('.clip'), bar = root.querySelector('.bar');
  var dragging = false;
  function move(x) {{
    var r = root.getBoundingClientRect();
    var pct = Math.max(0, Math.min(100, (x - r.left) / r.width * 100));
    bar.style.left = pct + '%';
    clip.style.clipPath = 'inset(0 0 0 ' + pct + '%)';
  }}
  root.addEventListener('mousedown', function(e) {{ dragging = true; move(e.clientX); e.preventDefault(); }});
  document.addEventListener('mouseup', function() {{ dragging = false; }});
  document.addEventListener('mousemove', function(e) {{ if (dragging) move(e.clientX); }});
  root.addEventListener('touchstart', function(e) {{ dragging = true; move(e.touches[0].clientX);
                                                     e.preventDefault(); }}, {{passive: false}});
  document.addEventListener('touchend', function() {{ dragging = false; }});
  document.addEventListener('touchmove', function(e) {{ if (dragging) move(e.touches[0].clientX); }},
                            {{passive: true}});
}})();
</script>"""
    display(HTML(html))


def _players(src, dst, label):
    """Two <video> tags for the same moment, when the files are small enough to live in the notebook body."""
    tags, skipped = [], []
    for p, name in ((src, 'source'), (dst, 'FlashVSR')):
        mb = os.path.getsize(p) / 1e6
        mime = VIDEO_MIME.get(os.path.splitext(p.lower())[1])
        if mb > EMBED_LIMIT_MB or not mime:
            skipped.append(f'{name} ({mb:.1f} MB' + (', a format the browser will not play inline'
                                                    if not mime else '') + ')')
            continue
        with open(p, 'rb') as fh:
            b64 = base64.b64encode(fh.read()).decode()
        tags.append(f'<div style="flex:1;min-width:0"><div style="font-size:12px;color:#777;margin:2px 0">'
                    f'{name} · {os.path.basename(p)}</div>'
                    f'<video src="data:{mime};base64,{b64}" controls muted loop playsinline '
                    f'style="width:100%;background:#000;border-radius:4px"></video></div>')
    html = (f'<div style="font-family:sans-serif;font-size:13px;margin-top:4px"><b>{label}</b> — '
            f'both files from {PREVIEW_POS:.0%} of the clip, playing</div>'
            '<div style="display:flex;gap:8px;align-items:flex-start;max-width:1200px;margin-bottom:20px">'
            + ''.join(tags) + '</div>')
    if skipped:
        html += ('<div style="font-family:sans-serif;font-size:12px;color:#8a6d3b;margin:-12px 0 18px 0">'
                 'not embedded: ' + ', '.join(skipped) + ' — the comparison above still shows it, and the '
                 'Download Results cell gives you the file.</div>')
    display(HTML(html))


for src, dst, label in _pair_list():
    try:
        after = _load(dst)
        before = _load(src) if src else None
        if before is None:                       # a result whose source is gone: show what there is
            print(f'{label} — the source file is no longer in {INPUT_DIR}, showing the result alone')
            display(resize_image_maintain_aspect(after, 1000))
        elif visualization_method == 'Slider':
            _slider(before, after, label)
        else:
            _side_by_side(before, after, label)
        if src and not (is_image_file(src) and is_image_file(dst)):
            _players(src, dst, label)            # a clip gets its two players whatever the comparison mode
    except Exception as e:
        print(f'Error processing {label}: {e}')

try:
    shutil.rmtree(VIS_TMP, ignore_errors=True)
except Exception:
    pass


In [ ]:
#@title ##**Download Results** { display-mode: "form" }
# title: Download Results
# One green button. Click -> the finished images/videos in /content/outputs are handed to the
# browser: through Colab's own files.download() when it exists (it streams the file, so big videos
# work), otherwise encoded into the page as a data URL. No file hosts, no extra buttons.
#
# A single file that fits CHUNK_MB is saved straight away; anything bigger (or a set of several
# files) is packed into zip parts so each auto-download stays manageable. A download is limited by
# how much base64 the browser tab can juggle — a 1 GB video is fine via the Output panel / "Save
# Version", but may fail through the browser; a higher CRF (or JPEG/WebP instead of PNG for images)
# in Config makes the files smaller if that happens.
# NOTE: the green color is forced via btn.style.button_color because Kaggle's theme ignores
# ipywidgets' standard button_style colors (the button otherwise looked black).
import os, base64, zipfile, time
import ipywidgets as widgets
from IPython.display import HTML, display

CHUNK_MB = 100          # max size of a single auto-download; bigger sets are split into parts
MIME = {'.mp4': 'video/mp4', '.webm': 'video/webm', '.mkv': 'video/x-matroska',
        '.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg', '.webp': 'image/webp'}

results = sorted(f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(tuple(MIME)))
assert results, 'No results — run the Run cell first.'

SPINNER = ('<style>.s2r-spin{display:inline-block;width:16px;height:16px;margin-right:8px;'
           'border:3px solid #c8e6c9;border-top-color:#4CAF50;border-radius:50%;'
           'animation:s2r-rot .8s linear infinite;vertical-align:middle}'
           '@keyframes s2r-rot{to{transform:rotate(360deg)}}</style>'
           '<div style="font-family:sans-serif;font-size:14px;margin:6px 0">'
           '<span class="s2r-spin"></span>MSG</div>')

def ok_html(msg):
    return f'<div style="font-family:sans-serif;font-size:14px;color:#2E7D32;margin:6px 0">{msg}</div>'

def err_html(msg):
    return f'<div style="font-family:sans-serif;font-size:14px;color:#B71C1C;margin:6px 0">{msg}</div>'

sizes = {f: os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in results}
total_mb = sum(sizes.values()) / 1e6
n_img = sum(1 for f in results if os.path.splitext(f)[1].lower() in ('.png', '.jpg', '.jpeg', '.webp'))
print(f'{len(results)} file(s) ({len(results) - n_img} video(s), {n_img} image(s)), '
      f'~{total_mb:.1f} MB total:')
for f in results:
    print(f'  {f} — {sizes[f] / 1e6:.1f} MB'
          + ('  (will be zipped in parts)' if sizes[f] / 1e6 > CHUNK_MB else ''))
print('(Individual files are also reachable without this cell: Files panel on the left → /content/outputs')
print(' → right-click → Download. And remember that /content dies with the runtime — take the zip before')
print(' you close the tab.)')

# ---------- UI: force the green color, Kaggle's theme ignores button_style ----------
btn = widgets.Button(description='Download', layout=widgets.Layout(width='320px', height='38px'))
btn.style.button_color = '#4CAF50'
try:
    btn.style.text_color = 'white'   # ipywidgets >= 8
except Exception:
    pass
status = widgets.HTML(value='')
js_out = widgets.Output()  # hidden slot used only to inject the auto-download <script>
display(HTML('<style>'
             '.widget-button { border-radius: 3px !important; color: white !important; '
             'font-weight: 600 !important; }'
             '</style>'))
display(widgets.VBox([btn, status, js_out]))


def trigger_download(path, name):
    """Colab's downloader when we are inside Colab; a data-URI <a> click everywhere else.

    files.download streams the file to the browser without putting base64 in the page, which is what a
    few-hundred-MB upscale survives; the fallback keeps the cell working in plain Jupyter and in tests.
    """
    try:
        from google.colab import files as _cf
        _cf.download(path, filename=name)
        return
    except Exception:
        pass
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = os.path.splitext(name)[1].lower()
    mime = 'application/zip' if name.endswith('.zip') else MIME.get(ext, 'application/octet-stream')
    js_out.outputs = ()
    js_out.append_display_data(HTML(
        f'<script>var a=document.createElement("a");a.download="{name}";'
        f'a.href="data:{mime};base64,{b64}";document.body.appendChild(a);a.click();a.remove();</script>'))


def plan_parts():
    """Group files into zip parts, each under CHUNK_MB. Single small file -> as-is."""
    paths = [os.path.join(OUTPUT_DIR, f) for f in results]
    if len(paths) == 1 and os.path.getsize(paths[0]) / 1e6 <= CHUNK_MB:
        return [('file', paths[0])]      # one small file: download it directly, no zip
    parts, cur, cur_mb = [], [], 0.0
    for p in paths:
        mb = os.path.getsize(p) / 1e6
        if cur and cur_mb + mb > CHUNK_MB:
            parts.append(cur); cur, cur_mb = [], 0.0
        cur.append(p); cur_mb += mb
    if cur:
        parts.append(cur)
    return [('zip', part) for part in parts]


def on_click(_):
    btn.disabled = True
    try:
        parts = plan_parts()
        if len(parts) == 1 and parts[0][0] == 'file':
            path = parts[0][1]
            name = os.path.basename(path)
            status.value = SPINNER.replace('MSG', f'Encoding {name}...')
            trigger_download(path, name)
            status.value = ok_html(f'Download of {name} started — check the browser downloads bar.')
            return
        n = len(parts)
        for i, (_, files) in enumerate(parts, 1):
            zname = 'flashvsr_results.zip' if n == 1 else f'flashvsr_results_part{i}of{n}.zip'
            zpath = os.path.join(os.path.dirname(OUTPUT_DIR.rstrip('/')), zname)   # next to outputs/
            status.value = SPINNER.replace('MSG', f'Packing {zname} ({i}/{n})...')
            if os.path.exists(zpath):
                os.remove(zpath)
            # ZIP_STORED: mp4/jpg are already compressed, DEFLATE would just burn CPU
            with zipfile.ZipFile(zpath, 'w', compression=zipfile.ZIP_STORED) as zf:
                for p in files:
                    zf.write(p, arcname=os.path.basename(p))
            status.value = SPINNER.replace('MSG', f'Starting download {i}/{n}: {zname}...')
            trigger_download(zpath, zname)
            if i < n:
                time.sleep(2)     # let the browser register the previous download
        status.value = ok_html(
            f'Started {n} download(s) — check the browser downloads bar.'
            + ('' if n == 1 else ' Unpack all parts into one folder.'))
    except Exception as e:
        status.value = err_html(f'Error: {e}')
    finally:
        btn.disabled = False


btn.on_click(on_click)
